# Data Feasibility: California Wildfire Pilot

## Purpose

This notebook checks whether the proposed data sources can support the SmokeWatch Replay project.

The pilot focuses on the 2021 Dixie Fire in California during July 13–August 31, 2021. We will examine whether fire detections, air-quality observations, weather records, and incident information can be cleaned and joined into hourly county-level records.

## Questions

1. Can each source be accessed and downloaded?
2. What fields and time formats does each source contain?
3. Can records be aligned by county and hour?
4. What missing values, duplicates, and timestamp problems must be handled?
5. Is the proposed pilot feasible for the project timeline?

## Data Sources to Validate

| Source | Purpose in the pipeline | Initial validation |
|---|---|---|
| NASA FIRMS | Fire detection events | Check location, date, time, and fire-confidence fields |
| EPA AirData | Hourly PM2.5 observations | Check PM2.5 values, station location, and timestamps |
| NOAA | Weather observations | Check temperature, wind, and observation time |
| CAL FIRE | Fire incident information | Check incident name, dates, counties, and coordinates |

The sources will be evaluated for accessibility, time coverage, geographic coverage, missing values, duplicate records, and compatibility with an hourly county-level join.

## Source 1: NASA FIRMS Active Fire Data

- **Official archive:** https://firms.modaps.eosdis.nasa.gov/download/
- **Area API:** https://firms.modaps.eosdis.nasa.gov/api/area/
- **Role:** Provides satellite-detected active-fire locations and observation times.
- **Initial product:** VIIRS NOAA-20 375 m Standard Processing (`VIIRS_NOAA20_SP`)
- **Temporal fit:** VIIRS NOAA-20 coverage begins on April 1, 2018, so it includes the 2021 Dixie Fire period.
- **Available formats:** CSV, JSON, and Shapefile.
- **Expected fields:** Latitude, longitude, acquisition date, acquisition time, confidence, fire radiative power, satellite, and day/night indicator.
- **Access requirement:** Historical archive access requires an Earthdata login or email verification. Programmatic API access requires a free FIRMS map key.
- **Initial decision:** Begin with the NOAA-20 VIIRS product to avoid duplicate detections from combining multiple satellites.

## Source 2: EPA AirData Hourly PM2.5

- **Official download page:** https://aqs.epa.gov/aqsweb/airdata/download_files.html
- **2021 hourly file:** https://aqs.epa.gov/aqsweb/airdata/hourly_88101_2021.zip
- **Parameter:** PM2.5 FRM/FEM Mass (`88101`)
- **Role:** Provides hourly PM2.5 measurements from outdoor air-quality monitors.
- **Format:** Compressed ZIP containing a CSV file with a header row.
- **Dataset size:** Approximately 8.6 million national records and 67 MB compressed.
- **Important fields:** State Code, County Code, Site Num, latitude, longitude, Date Local, Time Local, Date GMT, Time GMT, Sample Measurement, units, qualifier, and method information.
- **Planned filtering:** Keep California records for Butte, Plumas, Shasta, Lassen, and Tehama counties during July 13–August 31, 2021.
- **Timestamp plan:** Use the GMT date and time fields to align EPA observations with NASA FIRMS timestamps.
- **Initial concern:** A county may have multiple monitoring sites or instruments during the same hour. We must define an aggregation rule, such as calculating the hourly county mean after quality checks.

## Source 3: NOAA Global Hourly Weather Data

- **Official product page:** https://www.ncei.noaa.gov/products/land-based-station/integrated-surface-database
- **Data search:** https://www.ncei.noaa.gov/access/search/datasets/global-hourly/
- **API documentation:** https://www.ncei.noaa.gov/access/search/documentation/data-service/
- **Role:** Provides hourly and sub-hourly weather observations from ground stations.
- **Expected variables:** Temperature, dew point, wind speed, wind direction, wind gust, atmospheric pressure, visibility, and precipitation.
- **Time coverage:** Historical data include the 2021 Dixie Fire period.
- **Planned filtering:** Select weather stations in or near Butte, Plumas, Shasta, Lassen, and Tehama counties for July 13–August 31, 2021.
- **Timestamp plan:** Convert observation timestamps to UTC and round or aggregate them into one-hour windows.
- **Initial concern:** NOAA records are organized by weather station rather than directly by county. Station coordinates must be mapped to counties, and some counties may not contain a suitable station.
- **Initial decision:** Validate station availability before selecting the final weather variables or downloading the complete pilot period.

## Source 4: CAL FIRE Dixie Fire Incident Information

- **Official 2021 incident archive:** https://www.fire.ca.gov/incidents/2021/
- **Official Dixie Fire page:** https://www.fire.ca.gov/incidents/2021/7/13/dixie-fire/
- **Role:** Provides authoritative incident context and metadata for the pilot case study.
- **Available information:** Incident name, location, counties, acreage, containment, status, cause, structures affected, and archived updates.
- **Geographic scope:** Butte, Plumas, Shasta, Lassen, and Tehama counties.
- **Final reported size:** 963,309 acres.
- **Planned use:** Use CAL FIRE information as an incident reference or dimension record rather than as the primary hourly event stream.
- **Initial concern:** CAL FIRE updates are not consistently hourly, and some archived pages show differences in recorded start dates or county coverage as the incident evolved.
- **Initial decision:** Use NASA, EPA, and NOAA for hourly observations. Use CAL FIRE to validate the selected incident, dates, counties, and final incident characteristics.

## Preliminary Feasibility Assessment

| Source | Preliminary status | Main risk | Decision |
|---|---|---|---|
| NASA FIRMS | Feasible | API requests are limited to five-day batches and require a map key | Proceed with one VIIRS product and automate date batches |
| EPA AirData | Feasible | National hourly file is large and may contain multiple monitors per county-hour | Filter by date and county, then define an aggregation rule |
| NOAA Global Hourly | Conditionally feasible | Weather stations may not be available inside every selected county | Validate station coverage before finalizing the geographic join |
| CAL FIRE | Feasible for metadata | Incident updates are not consistently hourly | Use as an incident reference rather than an hourly stream |

## Preliminary Conclusion

The proposed pilot is technically feasible because all four sources cover the 2021 period and provide downloadable historical information. NASA FIRMS, EPA AirData, and NOAA can support the time-based pipeline, while CAL FIRE can provide incident context. The most important unresolved issue is whether suitable NOAA and EPA monitoring stations exist in or near all five selected counties.

## NASA FIRMS NOAA-20 Sample Acquisition

This section tests whether NOAA-20 VIIRS fire-detection data can be retrieved for California during the first five days of the Dixie Fire pilot period.

In [1]:
%pip install -q pandas requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os

import pandas as pd
import requests
from dotenv import load_dotenv

# Locate the repository root whether the notebook starts
# inside the repository or its notebooks folder.
current_folder = Path.cwd()
repo_root = (
    current_folder
    if (current_folder / ".env").exists()
    else current_folder.parent
)

env_path = repo_root / ".env"

assert env_path.exists(), ".env file was not found at the repository root."

load_dotenv(env_path)
firms_map_key = os.getenv("FIRMS_MAP_KEY")

assert firms_map_key, "FIRMS_MAP_KEY is missing from the .env file."

print("Setup complete: FIRMS key loaded securely.")
print("Repository root located successfully.")

Setup complete: FIRMS key loaded securely.
Repository root located successfully.


In [3]:
from io import StringIO

# Request settings: California, five days beginning July 13, 2021
source = "VIIRS_NOAA20_SP"
area = "-124.5,32.5,-114.0,42.1"
day_range = 5
start_date = "2021-07-13"

api_url = (
    "https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
    f"{firms_map_key}/{source}/{area}/{day_range}/{start_date}"
)

# Request the data without displaying the URL or MAP_KEY
try:
    response = requests.get(api_url, timeout=90)
except requests.RequestException:
    raise RuntimeError(
        "The NASA FIRMS request encountered a network error."
    ) from None

if response.status_code != 200:
    raise RuntimeError(
        f"NASA FIRMS returned status code {response.status_code}."
    )

firms_sample = pd.read_csv(StringIO(response.text))

required_columns = {"latitude", "longitude", "acq_date", "acq_time"}
missing_columns = required_columns - set(firms_sample.columns)

assert not missing_columns, (
    f"Expected FIRMS columns are missing: {missing_columns}"
)

# Save the retrieved data locally in an ignored folder
output_folder = repo_root / "data" / "raw" / "nasa_firms"
output_folder.mkdir(parents=True, exist_ok=True)

output_path = (
    output_folder
    / "viirs_noaa20_california_2021-07-13_5days.csv"
)

firms_sample.to_csv(output_path, index=False)

print("NASA FIRMS request completed successfully.")
print(f"Rows downloaded: {len(firms_sample):,}")
print(f"Columns received: {len(firms_sample.columns)}")
print(f"Date range: {firms_sample['acq_date'].min()} to "
      f"{firms_sample['acq_date'].max()}")
print(f"Saved locally to: {output_path.relative_to(repo_root)}")

display(firms_sample.head())

NASA FIRMS request completed successfully.
Rows downloaded: 2,956
Columns received: 15
Date range: 2021-07-13 to 2021-07-17
Saved locally to: data/raw/nasa_firms/viirs_noaa20_california_2021-07-13_5days.csv


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type
0,37.79095,-118.89727,296.97,0.73,0.76,2021-07-13,840,N20,VIIRS,n,2,284.78,4.54,N,0
1,37.79288,-118.88161,305.53,0.73,0.76,2021-07-13,840,N20,VIIRS,n,2,285.27,3.35,N,0
2,37.79301,-118.88929,309.68,0.73,0.76,2021-07-13,840,N20,VIIRS,n,2,286.95,3.70,N,0
3,37.79315,-118.89719,309.11,0.73,0.76,2021-07-13,840,N20,VIIRS,n,2,286.06,3.70,N,0
4,37.79779,-118.88048,312.54,0.73,0.76,2021-07-13,840,N20,VIIRS,n,2,287.36,5.56,N,0


### Initial NASA FIRMS Data-Quality Check

The downloaded sample is checked for its structure, missing values, duplicate rows, date coverage, and coordinate range before further filtering or transformation.

In [4]:
quality_summary = pd.DataFrame({
    "data_type": firms_sample.dtypes.astype(str),
    "missing_values": firms_sample.isna().sum(),
    "missing_percent": (
        firms_sample.isna().mean() * 100
    ).round(2)
})

print(f"Dataset shape: {firms_sample.shape}")
print(f"Duplicate rows: {firms_sample.duplicated().sum()}")
print(
    f"Unique acquisition dates: "
    f"{firms_sample['acq_date'].nunique()}"
)
print(
    f"Latitude range: {firms_sample['latitude'].min():.4f} "
    f"to {firms_sample['latitude'].max():.4f}"
)
print(
    f"Longitude range: {firms_sample['longitude'].min():.4f} "
    f"to {firms_sample['longitude'].max():.4f}"
)

display(quality_summary)

Dataset shape: (2956, 15)
Duplicate rows: 0
Unique acquisition dates: 5
Latitude range: 32.5014 to 41.4527
Longitude range: -123.6966 to -114.1916


,data_type,missing_values,missing_percent
latitude,float64,0,0.0
longitude,float64,0,0.0
bright_ti4,float64,0,0.0
scan,float64,0,0.0
track,float64,0,0.0
acq_date,object,0,0.0
acq_time,int64,0,0.0
satellite,object,0,0.0
instrument,object,0,0.0
confidence,object,0,0.0


### Create a Standard UTC Event Timestamp

NASA stores the acquisition date and time in separate columns. These fields are combined into one UTC timestamp so fire detections can later be grouped into hourly windows and joined with EPA and NOAA observations.

In [5]:
firms_clean = firms_sample.copy()

# Restore acquisition time to four digits.
# For example: 840 becomes 0840 and 5 becomes 0005.
firms_clean["acq_time_text"] = (
    firms_clean["acq_time"]
    .astype(str)
    .str.zfill(4)
)

# Combine the acquisition date and time as a UTC timestamp.
firms_clean["event_timestamp_utc"] = pd.to_datetime(
    firms_clean["acq_date"].astype(str)
    + " "
    + firms_clean["acq_time_text"],
    format="%Y-%m-%d %H%M",
    utc=True,
    errors="coerce"
)

invalid_timestamps = firms_clean[
    "event_timestamp_utc"
].isna().sum()

print(f"Invalid timestamps: {invalid_timestamps}")
print(
    "Earliest timestamp:",
    firms_clean["event_timestamp_utc"].min()
)
print(
    "Latest timestamp:",
    firms_clean["event_timestamp_utc"].max()
)

display(
    firms_clean[
        [
            "acq_date",
            "acq_time",
            "acq_time_text",
            "event_timestamp_utc"
        ]
    ].head()
)

Invalid timestamps: 0
Earliest timestamp: 2021-07-13 08:40:00+00:00
Latest timestamp: 2021-07-17 22:09:00+00:00


,acq_date,acq_time,acq_time_text,event_timestamp_utc
0,2021-07-13,840,0840,2021-07-13 08:40:00+00:00
1,2021-07-13,840,0840,2021-07-13 08:40:00+00:00
2,2021-07-13,840,0840,2021-07-13 08:40:00+00:00
3,2021-07-13,840,0840,2021-07-13 08:40:00+00:00
4,2021-07-13,840,0840,2021-07-13 08:40:00+00:00


### Inspect Fire-Detection Categories

The confidence, hotspot type, and day/night fields are examined before selecting records for the wildfire pilot. This avoids removing records without first understanding their meaning and frequency.

In [6]:
confidence_counts = (
    firms_clean["confidence"]
    .value_counts(dropna=False)
    .rename_axis("confidence")
    .reset_index(name="record_count")
)

type_counts = (
    firms_clean["type"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("type")
    .reset_index(name="record_count")
)

daynight_counts = (
    firms_clean["daynight"]
    .value_counts(dropna=False)
    .rename_axis("daynight")
    .reset_index(name="record_count")
)

print("Confidence categories:")
display(confidence_counts)

print("Hotspot types:")
display(type_counts)

print("Day/night categories:")
display(daynight_counts)

Confidence categories:


,confidence,record_count
0,n,2330
1,h,354
2,l,272


Hotspot types:


,type,record_count
0,0,2860
1,2,96


Day/night categories:


,daynight,record_count
0,N,1870
1,D,1086


### NASA FIRMS Filtering Decision

For the primary wildfire analysis, only presumed vegetation fires (`type = 0`) with nominal (`n`) or high (`h`) confidence are retained. Other static land sources and low-confidence detections are excluded to reduce unrelated heat sources and potential false positives. Both daytime and nighttime detections are retained.

In [7]:
firms_filtered = firms_clean[
    (firms_clean["type"] == 0)
    & (firms_clean["confidence"].isin(["n", "h"]))
].copy()

# Create the hourly key required for later integration.
firms_filtered["event_hour_utc"] = (
    firms_filtered["event_timestamp_utc"].dt.floor("h")
)

records_removed = len(firms_clean) - len(firms_filtered)
percent_retained = (
    len(firms_filtered) / len(firms_clean) * 100
)

print(f"Original records: {len(firms_clean):,}")
print(f"Filtered records: {len(firms_filtered):,}")
print(f"Records removed: {records_removed:,}")
print(f"Percentage retained: {percent_retained:.2f}%")
print(
    f"Unique hourly windows: "
    f"{firms_filtered['event_hour_utc'].nunique()}"
)

display(
    firms_filtered[
        [
            "event_timestamp_utc",
            "event_hour_utc",
            "latitude",
            "longitude",
            "confidence",
            "type",
            "frp",
            "daynight"
        ]
    ].head()
)

Original records: 2,956
Filtered records: 2,591
Records removed: 365
Percentage retained: 87.65%
Unique hourly windows: 17


,event_timestamp_utc,event_hour_utc,latitude,longitude,confidence,type,frp,daynight
0,2021-07-13 08:40:00+00:00,2021-07-13 08:00:00+00:00,37.79095,-118.89727,n,0,4.54,N
1,2021-07-13 08:40:00+00:00,2021-07-13 08:00:00+00:00,37.79288,-118.88161,n,0,3.35,N
2,2021-07-13 08:40:00+00:00,2021-07-13 08:00:00+00:00,37.79301,-118.88929,n,0,3.70,N
3,2021-07-13 08:40:00+00:00,2021-07-13 08:00:00+00:00,37.79315,-118.89719,n,0,3.70,N
4,2021-07-13 08:40:00+00:00,2021-07-13 08:00:00+00:00,37.79779,-118.88048,n,0,5.56,N


In [8]:
%pip install -q geopandas pyogrio

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
Note: you may need to restart the kernel to use updated packages.


In [9]:
import geopandas as gpd
import pyogrio

print("GeoPandas version:", gpd.__version__)
print("Pyogrio version:", pyogrio.__version__)
print("Geographic packages imported successfully.")

GeoPandas version: 1.2.0
Pyogrio version: 0.13.0
Geographic packages imported successfully.


### County Boundary Reference Data

Official 2021 U.S. Census cartographic boundary data is used to map each NASA FIRMS latitude and longitude to a California county. Only Butte, Plumas, Shasta, Lassen, and Tehama counties are retained for the pilot.

In [10]:
county_boundary_url = (
    "https://www2.census.gov/geo/tiger/GENZ2021/shp/"
    "cb_2021_us_county_500k.zip"
)

reference_folder = repo_root / "data" / "raw" / "reference"
reference_folder.mkdir(parents=True, exist_ok=True)

county_zip_path = (
    reference_folder / "cb_2021_us_county_500k.zip"
)

# Download once and reuse the local copy in later notebook runs.
if not county_zip_path.exists():
    boundary_response = requests.get(
        county_boundary_url,
        timeout=120
    )

    if boundary_response.status_code != 200:
        raise RuntimeError(
            "The Census county-boundary download failed with "
            f"status code {boundary_response.status_code}."
        )

    county_zip_path.write_bytes(boundary_response.content)
    print("County-boundary file downloaded.")
else:
    print("Using the existing local county-boundary file.")

county_boundaries = gpd.read_file(
    f"zip://{county_zip_path}"
)

pilot_county_names = [
    "Butte",
    "Plumas",
    "Shasta",
    "Lassen",
    "Tehama"
]

pilot_counties = county_boundaries[
    (county_boundaries["STATEFP"] == "06")
    & (county_boundaries["NAME"].isin(pilot_county_names))
].copy()

assert set(pilot_counties["NAME"]) == set(
    pilot_county_names
), "One or more pilot counties were not found."

print(f"Boundary CRS: {pilot_counties.crs}")
print(f"Pilot counties found: {len(pilot_counties)}")

display(
    pilot_counties[
        ["STATEFP", "COUNTYFP", "GEOID", "NAME"]
    ].sort_values("NAME")
)

County-boundary file downloaded.
Boundary CRS: EPSG:4269
Pilot counties found: 5


,STATEFP,COUNTYFP,GEOID,NAME
551,06,007,06007,Butte
2573,06,035,06035,Lassen
94,06,063,06063,Plumas
807,06,089,06089,Shasta
802,06,103,06103,Tehama


### Assign Fire Detections to Pilot Counties

Each filtered NASA FIRMS coordinate is converted into a geographic point and spatially joined with the Census county boundaries. This adds an exact county name and county GEOID to each matched detection.

In [11]:
# Convert FIRMS latitude and longitude into geographic points.
firms_points = gpd.GeoDataFrame(
    firms_filtered.copy(),
    geometry=gpd.points_from_xy(
        firms_filtered["longitude"],
        firms_filtered["latitude"]
    ),
    crs="EPSG:4326"
)

# Match the coordinate reference system used by the boundaries.
firms_points = firms_points.to_crs(
    pilot_counties.crs
)

# Assign each point to one of the five pilot counties.
firms_pilot = gpd.sjoin(
    firms_points,
    pilot_counties[
        ["GEOID", "NAME", "geometry"]
    ],
    how="inner",
    predicate="within"
)

firms_pilot = (
    firms_pilot
    .drop(columns=["index_right"])
    .rename(
        columns={
            "GEOID": "county_geoid",
            "NAME": "county"
        }
    )
)

county_counts = (
    firms_pilot["county"]
    .value_counts()
    .rename_axis("county")
    .reset_index(name="fire_detection_count")
)

print(
    f"Statewide filtered records: "
    f"{len(firms_filtered):,}"
)
print(
    f"Records inside pilot counties: "
    f"{len(firms_pilot):,}"
)
print(
    f"Pilot counties with detections: "
    f"{firms_pilot['county'].nunique()}"
)

display(county_counts)

Statewide filtered records: 2,591
Records inside pilot counties: 1,209
Pilot counties with detections: 4


,county,fire_detection_count
0,Plumas,1085
1,Lassen,113
2,Butte,6
3,Tehama,5


### Feasibility Finding: County and Date Are Not Sufficient

The first five-day sample contains 1,209 fire detections within the five pilot counties, but the distribution suggests that other fires are also present. Therefore, county and date alone cannot identify Dixie Fire observations. An official CAL FIRE incident perimeter will be used as an additional spatial filter.

The available CAL FIRE perimeter represents the completed fire boundary rather than its exact daily progression. It is therefore suitable as an incident-area mask, but this limitation must be documented.

In [12]:
calfire_query_url = (
    "https://services.arcgis.com/V6ZHFr6zdgNZuVG0/"
    "ArcGIS/rest/services/California_Fire_Perimeters/"
    "FeatureServer/0/query"
)

query_parameters = {
    "where": "YEAR_ = 2021 AND FIRE_NAME = 'DIXIE'",
    "outFields": (
        "OBJECTID,YEAR_,FIRE_NAME,GIS_ACRES,"
        "ALARM_DATE,CONT_DATE"
    ),
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "geojson"
}

perimeter_response = requests.get(
    calfire_query_url,
    params=query_parameters,
    timeout=120
)

if perimeter_response.status_code != 200:
    raise RuntimeError(
        "CAL FIRE perimeter request failed with status "
        f"{perimeter_response.status_code}."
    )

perimeter_data = perimeter_response.json()

if "error" in perimeter_data:
    raise RuntimeError(
        f"CAL FIRE API error: {perimeter_data['error']}"
    )

dixie_perimeter = gpd.GeoDataFrame.from_features(
    perimeter_data["features"],
    crs="EPSG:4326"
)

assert not dixie_perimeter.empty, (
    "No 2021 Dixie Fire perimeter was returned."
)

print(
    f"Dixie perimeter records returned: "
    f"{len(dixie_perimeter)}"
)

display(
    dixie_perimeter[
        [
            "YEAR_",
            "FIRE_NAME",
            "GIS_ACRES",
            "ALARM_DATE",
            "CONT_DATE"
        ]
    ]
)

Dixie perimeter records returned: 1


,YEAR_,FIRE_NAME,GIS_ACRES,ALARM_DATE,CONT_DATE
0,2021,DIXIE,963405.4,1626134400000,1635120000000


### Isolate the Dixie Fire Incident Area

The final CAL FIRE Dixie Fire perimeter is used as an incident-area mask. A one-kilometer buffer is added to account for VIIRS pixel size and small differences between satellite coordinates and the generalized perimeter boundary.

This is a spatial approximation. Because the final perimeter includes areas burned later in the incident, it does not represent the fire's exact daily progression.

In [13]:
# Convert CAL FIRE date values from milliseconds to readable UTC dates.
for date_column in ["ALARM_DATE", "CONT_DATE"]:
    dixie_perimeter[date_column] = pd.to_datetime(
        dixie_perimeter[date_column],
        unit="ms",
        utc=True,
        errors="coerce"
    )

# Save the original CAL FIRE perimeter locally.
dixie_perimeter_path = (
    reference_folder
    / "calfire_dixie_2021_final_perimeter.geojson"
)

dixie_perimeter.to_file(
    dixie_perimeter_path,
    driver="GeoJSON"
)

# Use California Albers so the buffer can be measured in meters.
projected_crs = "EPSG:3310"

dixie_projected = dixie_perimeter.to_crs(projected_crs)
pilot_points_projected = firms_pilot.to_crs(projected_crs)

# Combine the perimeter geometry and add a 1-kilometer buffer.
dixie_buffered_geometry = (
    dixie_projected.geometry.union_all().buffer(1000)
)

dixie_mask = gpd.GeoDataFrame(
    {"incident": ["Dixie Fire"]},
    geometry=[dixie_buffered_geometry],
    crs=projected_crs
)

# Retain only detections inside the buffered Dixie perimeter.
firms_dixie = gpd.sjoin(
    pilot_points_projected,
    dixie_mask,
    how="inner",
    predicate="within"
).drop(columns=["index_right"])

dixie_county_counts = (
    firms_dixie["county"]
    .value_counts()
    .rename_axis("county")
    .reset_index(name="fire_detection_count")
)

print(
    f"Pilot-county records before perimeter filter: "
    f"{len(firms_pilot):,}"
)
print(
    f"Dixie-area records after perimeter filter: "
    f"{len(firms_dixie):,}"
)
print(
    f"Counties represented: "
    f"{firms_dixie['county'].nunique()}"
)
print(
    "Perimeter dates:",
    dixie_perimeter["ALARM_DATE"].min(),
    "to",
    dixie_perimeter["CONT_DATE"].max()
)

display(dixie_county_counts)

Pilot-county records before perimeter filter: 1,209
Dixie-area records after perimeter filter: 976
Counties represented: 3
Perimeter dates: 2021-07-13 00:00:00+00:00 to 2021-10-25 00:00:00+00:00


,county,fire_detection_count
0,Plumas,964
1,Butte,6
2,Lassen,6


### Validate the Daily Geographic Pattern

The perimeter-filtered detections are summarized by date and county. This check helps determine whether the records show a reasonable temporal progression rather than an unexplained concentration from another incident.

In [14]:
daily_county_counts = (
    firms_dixie
    .groupby(["acq_date", "county"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=pilot_county_names, fill_value=0)
)

daily_county_counts["daily_total"] = (
    daily_county_counts.sum(axis=1)
)

print("Dixie-area detections by date and county:")
display(daily_county_counts)

print(
    "Total filtered Dixie-area records:",
    f"{daily_county_counts['daily_total'].sum():,}"
)

Dixie-area detections by date and county:


county,Butte,Plumas,Shasta,Lassen,Tehama,daily_total
acq_date,,,,,,
2021-07-13,0,46,0,0,0,46
2021-07-14,0,98,0,0,0,98
2021-07-15,4,171,0,6,0,181
2021-07-16,2,268,0,0,0,270
2021-07-17,0,381,0,0,0,381


Total filtered Dixie-area records: 976


### Create the Hourly County-Level Fire Dataset

The incident-area detections are aggregated by county and UTC hour. A complete hourly grid is created for all five pilot counties so that hours without a fire detection are represented with a count of zero rather than being absent from the dataset.

Mean and maximum FRP remain missing when no fire was detected because an average or maximum does not exist for an empty group.

In [15]:
observed_hourly_fire = (
    firms_dixie
    .groupby(
        ["event_hour_utc", "county"],
        observed=True
    )
    .agg(
        fire_detection_count=("latitude", "size"),
        total_frp_mw=("frp", "sum"),
        mean_frp_mw=("frp", "mean"),
        max_frp_mw=("frp", "max"),
        high_confidence_count=(
            "confidence",
            lambda values: (values == "h").sum()
        ),
        night_detection_count=(
            "daynight",
            lambda values: (values == "N").sum()
        )
    )
    .reset_index()
)

# Create every UTC hour for all five counties.
all_hours = pd.date_range(
    start="2021-07-13 00:00:00",
    end="2021-07-17 23:00:00",
    freq="h",
    tz="UTC"
)

complete_hourly_grid = (
    pd.MultiIndex.from_product(
        [all_hours, pilot_county_names],
        names=["event_hour_utc", "county"]
    )
    .to_frame(index=False)
)

hourly_fire_sample = complete_hourly_grid.merge(
    observed_hourly_fire,
    on=["event_hour_utc", "county"],
    how="left"
)

count_columns = [
    "fire_detection_count",
    "high_confidence_count",
    "night_detection_count"
]

hourly_fire_sample[count_columns] = (
    hourly_fire_sample[count_columns]
    .fillna(0)
    .astype(int)
)

hourly_fire_sample["total_frp_mw"] = (
    hourly_fire_sample["total_frp_mw"].fillna(0)
)

# Save the cleaned hourly sample locally.
interim_folder = (
    repo_root / "data" / "interim" / "nasa_firms"
)
interim_folder.mkdir(parents=True, exist_ok=True)

hourly_output_path = (
    interim_folder
    / "dixie_fire_hourly_county_sample_2021-07-13_17.csv"
)

hourly_fire_sample.to_csv(
    hourly_output_path,
    index=False
)

print(f"Hourly county rows: {len(hourly_fire_sample):,}")
print(
    "Rows with one or more detections:",
    (
        hourly_fire_sample["fire_detection_count"] > 0
    ).sum()
)
print(
    "Total detections preserved:",
    hourly_fire_sample["fire_detection_count"].sum()
)
print(
    "Saved locally to:",
    hourly_output_path.relative_to(repo_root)
)

display(
    hourly_fire_sample[
        hourly_fire_sample["fire_detection_count"] > 0
    ].head(10)
)

Hourly county rows: 600
Rows with one or more detections: 19
Total detections preserved: 976
Saved locally to: data/interim/nasa_firms/dixie_fire_hourly_county_sample_2021-07-13_17.csv


,event_hour_utc,county,fire_detection_count,total_frp_mw,mean_frp_mw,max_frp_mw,high_confidence_count,night_detection_count
51,2021-07-13 10:00:00+00:00,Plumas,16,47.98,2.998750,10.45,0,16
101,2021-07-13 20:00:00+00:00,Plumas,20,541.19,27.059500,97.71,6,0
106,2021-07-13 21:00:00+00:00,Plumas,10,533.12,53.312000,146.02,5,0
171,2021-07-14 10:00:00+00:00,Plumas,44,264.24,6.005455,34.91,1,44
216,2021-07-14 19:00:00+00:00,Plumas,7,192.69,27.527143,51.59,2,0
226,2021-07-14 21:00:00+00:00,Plumas,47,3735.42,79.477021,558.40,18,0
285,2021-07-15 09:00:00+00:00,Butte,4,6.15,1.537500,2.40,0,4
286,2021-07-15 09:00:00+00:00,Plumas,96,1107.82,11.539792,78.11,1,96
288,2021-07-15 09:00:00+00:00,Lassen,1,35.54,35.540000,35.54,0,1
346,2021-07-15 21:00:00+00:00,Plumas,75,5351.45,71.352667,263.74,34,0


## NASA FIRMS Feasibility Result

NASA FIRMS is feasible for the SmokeWatch Replay project.

The five-day NOAA-20 sample contained 2,956 California fire detections. After removing low-confidence detections and non-vegetation heat sources, 2,591 records remained. A county spatial join identified 1,209 records within the five pilot counties.

County and date filtering alone was not sufficient to isolate the Dixie Fire because other wildfire activity was present in the region. Applying the official CAL FIRE Dixie Fire perimeter with a one-kilometer buffer reduced the sample to 976 incident-area detections.

All acquisition dates and times were converted successfully into UTC timestamps. The detections were then aggregated into a complete dataset of 600 county-hour rows, representing five counties across 120 hours. Hours without detections were explicitly recorded with a count of zero.

### Limitations

- NASA satellite detections occur during satellite passes rather than continuously every hour.
- The CAL FIRE boundary represents the final fire perimeter, not its exact daily progression.
- The one-kilometer perimeter buffer is an analytical assumption used to accommodate satellite pixel size and boundary uncertainty.
- The five-day sample establishes feasibility but does not yet represent the complete July 13–August 31 pilot period.

### Decision

Proceed with NASA FIRMS as the fire-detection source. The full pilot period can be retrieved in five-day API batches and processed using the validated cleaning, spatial-filtering, and hourly-aggregation steps.

## EPA AirData PM2.5 Sample Acquisition

EPA AirData provides hourly PM2.5 FRM/FEM measurements under parameter code 88101. The 2021 national file contains more than 8.6 million records. It will be downloaded locally and processed in chunks so that only the pilot counties and dates are retained in memory.

In [16]:
from zipfile import ZipFile

epa_file_url = (
    "https://aqs.epa.gov/aqsweb/airdata/"
    "hourly_88101_2021.zip"
)

epa_raw_folder = (
    repo_root / "data" / "raw" / "epa_airdata"
)
epa_raw_folder.mkdir(parents=True, exist_ok=True)

epa_zip_path = (
    epa_raw_folder / "hourly_88101_2021.zip"
)

# Download once and reuse the local file.
if not epa_zip_path.exists():
    with requests.get(
        epa_file_url,
        stream=True,
        timeout=300
    ) as epa_response:
        if epa_response.status_code != 200:
            raise RuntimeError(
                "EPA download failed with status "
                f"{epa_response.status_code}."
            )

        with open(epa_zip_path, "wb") as output_file:
            for data_chunk in epa_response.iter_content(
                chunk_size=1024 * 1024
            ):
                if data_chunk:
                    output_file.write(data_chunk)

    print("EPA file downloaded successfully.")
else:
    print("Using the existing local EPA file.")

file_size_mb = (
    epa_zip_path.stat().st_size / (1024 * 1024)
)

with ZipFile(epa_zip_path) as epa_zip:
    zip_members = epa_zip.namelist()

print(f"Downloaded size: {file_size_mb:.2f} MB")
print("Files inside ZIP:", zip_members)
print(
    "Saved locally to:",
    epa_zip_path.relative_to(repo_root)
)

EPA file downloaded successfully.
Downloaded size: 65.68 MB
Files inside ZIP: ['hourly_88101_2021.csv']
Saved locally to: data/raw/epa_airdata/hourly_88101_2021.zip


### Inspect the EPA Hourly PM2.5 Schema

A small preview is read from the compressed file to verify column names, date fields, geographic identifiers, measurements, qualifiers, and method information before chunk-based filtering.

In [17]:
epa_csv_name = zip_members[0]

with ZipFile(epa_zip_path) as epa_zip:
    with epa_zip.open(epa_csv_name) as epa_csv:
        epa_preview = pd.read_csv(
            epa_csv,
            nrows=5,
            dtype=str
        )

print(f"EPA columns received: {len(epa_preview.columns)}")
print("Column names:")
print(epa_preview.columns.tolist())

preview_columns = [
    "State Code",
    "County Code",
    "Site Num",
    "Latitude",
    "Longitude",
    "Date GMT",
    "Time GMT",
    "Sample Measurement",
    "Units of Measure",
    "Qualifier",
    "Method Code",
    "County Name"
]

available_preview_columns = [
    column
    for column in preview_columns
    if column in epa_preview.columns
]

display(epa_preview[available_preview_columns])

EPA columns received: 24
Column names:
['State Code', 'County Code', 'Site Num', 'Parameter Code', 'POC', 'Latitude', 'Longitude', 'Datum', 'Parameter Name', 'Date Local', 'Time Local', 'Date GMT', 'Time GMT', 'Sample Measurement', 'Units of Measure', 'MDL', 'Uncertainty', 'Qualifier', 'Method Type', 'Method Code', 'Method Name', 'State Name', 'County Name', 'Date of Last Change']


,State Code,County Code,Site Num,Latitude,Longitude,Date GMT,Time GMT,Sample Measurement,Units of Measure,Qualifier,Method Code,County Name
0,01,055,0010,33.98821,-85.992556,2021-12-07,21:00,5,Micrograms/cubic meter (LC),NaN,209,Etowah
1,01,055,0010,33.98821,-85.992556,2021-12-07,22:00,3,Micrograms/cubic meter (LC),NaN,209,Etowah
2,01,055,0010,33.98821,-85.992556,2021-12-07,23:00,7,Micrograms/cubic meter (LC),NaN,209,Etowah
3,01,055,0010,33.98821,-85.992556,2021-12-08,00:00,5,Micrograms/cubic meter (LC),NaN,209,Etowah
4,01,055,0010,33.98821,-85.992556,2021-12-08,01:00,6,Micrograms/cubic meter (LC),NaN,209,Etowah


### Filter EPA Data in Chunks

The national EPA file is processed in chunks to avoid loading more than 8.6 million rows into memory simultaneously. Only California records from the five pilot counties and the five-day feasibility period are retained.

Following EPA's PM2.5 advisory, original T640 and T640X measurements with method codes 236 and 238 are excluded because corrected or network-aligned versions are available.

In [18]:
epa_columns = [
    "State Code",
    "County Code",
    "Site Num",
    "Parameter Code",
    "POC",
    "Latitude",
    "Longitude",
    "Date GMT",
    "Time GMT",
    "Sample Measurement",
    "Units of Measure",
    "Qualifier",
    "Method Code",
    "Method Name",
    "County Name"
]

epa_code_types = {
    "State Code": str,
    "County Code": str,
    "Site Num": str,
    "Parameter Code": str,
    "POC": str,
    "Method Code": str
}

pilot_county_codes = set(
    pilot_counties["COUNTYFP"].astype(str)
)

epa_filtered_chunks = []
epa_rows_scanned = 0

with ZipFile(epa_zip_path) as epa_zip:
    with epa_zip.open(epa_csv_name) as epa_csv:
        chunk_reader = pd.read_csv(
            epa_csv,
            usecols=epa_columns,
            dtype=epa_code_types,
            chunksize=250_000,
            low_memory=False
        )

        for epa_chunk in chunk_reader:
            epa_rows_scanned += len(epa_chunk)

            matching_rows = (
                (epa_chunk["State Code"] == "06")
                & (
                    epa_chunk["County Code"]
                    .isin(pilot_county_codes)
                )
                & (
                    epa_chunk["Date GMT"]
                    .between(
                        "2021-07-13",
                        "2021-07-17"
                    )
                )
                & (
                    ~epa_chunk["Method Code"]
                    .isin(["236", "238"])
                )
            )

            filtered_chunk = epa_chunk.loc[
                matching_rows
            ].copy()

            if not filtered_chunk.empty:
                epa_filtered_chunks.append(filtered_chunk)

epa_sample = pd.concat(
    epa_filtered_chunks,
    ignore_index=True
)

print(f"National rows scanned: {epa_rows_scanned:,}")
print(f"Pilot records retained: {len(epa_sample):,}")
print(
    f"Counties represented: "
    f"{epa_sample['County Name'].nunique()}"
)
print(
    f"Date range: {epa_sample['Date GMT'].min()} "
    f"to {epa_sample['Date GMT'].max()}"
)

display(
    epa_sample["County Name"]
    .value_counts()
    .rename_axis("county")
    .reset_index(name="pm25_record_count")
)

National rows scanned: 8,620,507
Pilot records retained: 238
Counties represented: 2
Date range: 2021-07-13 to 2021-07-17


,county,pm25_record_count
0,Tehama,120
1,Butte,118


### EPA Coverage Gap and Secondary PM2.5 Product

The FRM/FEM dataset (`88101`) contains records only for Butte and Tehama counties during the sample period. To determine whether the remaining counties have acceptable continuous PM2.5 monitoring, EPA parameter `88502` will also be evaluated.

The two products will not be blindly averaged together. The final aggregation will prioritize `88101` when available and use `88502` only as supplementary coverage.

In [19]:
epa_88502_url = (
    "https://aqs.epa.gov/aqsweb/airdata/"
    "hourly_88502_2021.zip"
)

epa_88502_zip_path = (
    epa_raw_folder / "hourly_88502_2021.zip"
)

if not epa_88502_zip_path.exists():
    with requests.get(
        epa_88502_url,
        stream=True,
        timeout=300
    ) as epa_88502_response:
        if epa_88502_response.status_code != 200:
            raise RuntimeError(
                "EPA 88502 download failed with status "
                f"{epa_88502_response.status_code}."
            )

        with open(
            epa_88502_zip_path,
            "wb"
        ) as output_file:
            for data_chunk in (
                epa_88502_response.iter_content(
                    chunk_size=1024 * 1024
                )
            ):
                if data_chunk:
                    output_file.write(data_chunk)

    print("EPA 88502 file downloaded successfully.")
else:
    print("Using the existing local EPA 88502 file.")

with ZipFile(epa_88502_zip_path) as epa_88502_zip:
    epa_88502_members = epa_88502_zip.namelist()

print(
    "Downloaded size:",
    f"{epa_88502_zip_path.stat().st_size / (1024 ** 2):.2f} MB"
)
print("Files inside ZIP:", epa_88502_members)

EPA 88502 file downloaded successfully.
Downloaded size: 18.32 MB
Files inside ZIP: ['hourly_88502_2021.csv']


In [20]:
epa_88502_csv_name = epa_88502_members[0]

epa_88502_filtered_chunks = []
epa_88502_rows_scanned = 0

with ZipFile(epa_88502_zip_path) as epa_88502_zip:
    with epa_88502_zip.open(
        epa_88502_csv_name
    ) as epa_88502_csv:
        chunk_reader = pd.read_csv(
            epa_88502_csv,
            usecols=epa_columns,
            dtype=epa_code_types,
            chunksize=250_000,
            low_memory=False
        )

        for epa_chunk in chunk_reader:
            epa_88502_rows_scanned += len(epa_chunk)

            matching_rows = (
                (epa_chunk["State Code"] == "06")
                & (
                    epa_chunk["County Code"]
                    .isin(pilot_county_codes)
                )
                & (
                    epa_chunk["Date GMT"]
                    .between(
                        "2021-07-13",
                        "2021-07-17"
                    )
                )
            )

            filtered_chunk = epa_chunk.loc[
                matching_rows
            ].copy()

            if not filtered_chunk.empty:
                epa_88502_filtered_chunks.append(
                    filtered_chunk
                )

epa_88502_sample = pd.concat(
    epa_88502_filtered_chunks,
    ignore_index=True
)

print(
    f"National 88502 rows scanned: "
    f"{epa_88502_rows_scanned:,}"
)
print(
    f"Pilot 88502 records retained: "
    f"{len(epa_88502_sample):,}"
)
print(
    f"Counties represented: "
    f"{epa_88502_sample['County Name'].nunique()}"
)
print(
    f"Date range: "
    f"{epa_88502_sample['Date GMT'].min()} "
    f"to {epa_88502_sample['Date GMT'].max()}"
)

display(
    epa_88502_sample["County Name"]
    .value_counts()
    .rename_axis("county")
    .reset_index(name="pm25_record_count")
)

National 88502 rows scanned: 2,397,085
Pilot 88502 records retained: 703
Counties represented: 3
Date range: 2021-07-13 to 2021-07-17


,county,pm25_record_count
0,Plumas,351
1,Butte,232
2,Shasta,120


### Combined EPA Coverage Decision

Combining EPA parameters 88101 and 88502 provides direct PM2.5 monitoring for Butte, Plumas, Shasta, and Tehama counties. Lassen County has no records in either product during the sample period and will remain missing.

Parameter 88101 will be preferred when it is available for a county-hour. Parameter 88502 will provide supplementary continuous-monitor coverage. Measurements will later be averaged first by site and then by county so sites with multiple instruments do not receive extra weight.

In [21]:
epa_88101_sample = epa_sample.copy()
epa_88101_sample["pm25_product"] = "88101"

epa_88502_sample = epa_88502_sample.copy()
epa_88502_sample["pm25_product"] = "88502"

epa_combined = pd.concat(
    [epa_88101_sample, epa_88502_sample],
    ignore_index=True
)

# Create one UTC timestamp.
epa_combined["event_timestamp_utc"] = pd.to_datetime(
    epa_combined["Date GMT"].astype(str)
    + " "
    + epa_combined["Time GMT"].astype(str),
    format="%Y-%m-%d %H:%M",
    utc=True,
    errors="coerce"
)

# Convert measurements to numeric values.
epa_combined["pm25_ug_m3"] = pd.to_numeric(
    epa_combined["Sample Measurement"],
    errors="coerce"
)

# Identify a physical monitoring site.
epa_combined["site_id"] = (
    epa_combined["State Code"]
    + "-"
    + epa_combined["County Code"]
    + "-"
    + epa_combined["Site Num"]
)

coverage_table = pd.crosstab(
    epa_combined["County Name"],
    epa_combined["pm25_product"]
).reindex(
    index=pilot_county_names,
    columns=["88101", "88502"],
    fill_value=0
)

print(f"Combined records: {len(epa_combined):,}")
print(
    "Invalid timestamps:",
    epa_combined["event_timestamp_utc"].isna().sum()
)
print(
    "Missing PM2.5 measurements:",
    epa_combined["pm25_ug_m3"].isna().sum()
)
print(
    "Exact duplicate rows:",
    epa_combined.duplicated().sum()
)
print(
    "Unique monitoring sites:",
    epa_combined["site_id"].nunique()
)

print("Coverage by county and EPA product:")
display(coverage_table)

print("Qualifier frequencies:")
display(
    epa_combined["Qualifier"]
    .fillna("No qualifier")
    .value_counts()
    .rename_axis("qualifier")
    .reset_index(name="record_count")
)

Combined records: 941
Invalid timestamps: 0
Missing PM2.5 measurements: 0
Exact duplicate rows: 0
Unique monitoring sites: 8
Coverage by county and EPA product:


pm25_product,88101,88502
County Name,,
Butte,118,232
Plumas,0,351
Shasta,0,120
Lassen,0,0
Tehama,120,0


Qualifier frequencies:


,qualifier,record_count
0,No qualifier,731
1,IT,210


### Create the Hourly County-Level PM2.5 Dataset

EPA observations are first averaged by monitoring site and UTC hour. The site-level measurements are then averaged by county so that a location with multiple instruments does not receive additional weight.

When both EPA products exist for the same county-hour, parameter 88101 is preferred. Parameter 88502 supplies additional continuous-monitor coverage when 88101 is unavailable. Qualifier `IT` is retained because it identifies U.S. wildfire influence rather than invalid data. Lassen County remains missing because neither EPA product contains a direct observation.

In [22]:
epa_clean = epa_combined[
    epa_combined["event_timestamp_utc"].notna()
    & epa_combined["pm25_ug_m3"].notna()
    & (
        ~epa_combined["Method Code"]
        .isin(["236", "238"])
    )
].copy()

epa_clean["event_hour_utc"] = (
    epa_clean["event_timestamp_utc"].dt.floor("h")
)

epa_clean["wildfire_qualifier"] = (
    epa_clean["Qualifier"].fillna("").eq("IT")
)

# Stage 1: average instruments at the same site and hour.
epa_site_hour = (
    epa_clean
    .groupby(
        [
            "event_hour_utc",
            "County Name",
            "site_id",
            "pm25_product"
        ],
        as_index=False
    )
    .agg(
        pm25_site_hour_ug_m3=("pm25_ug_m3", "mean"),
        instrument_count=("POC", "nunique"),
        wildfire_qualifier=(
            "wildfire_qualifier",
            "max"
        )
    )
)

# Stage 2: average sites within each county and product.
epa_county_product_hour = (
    epa_site_hour
    .groupby(
        [
            "event_hour_utc",
            "County Name",
            "pm25_product"
        ],
        as_index=False
    )
    .agg(
        pm25_ug_m3=("pm25_site_hour_ug_m3", "mean"),
        monitor_site_count=("site_id", "nunique"),
        wildfire_flagged_site_count=(
            "wildfire_qualifier",
            "sum"
        )
    )
)

# Prefer 88101 over 88502 when both exist.
epa_county_product_hour["source_priority"] = (
    epa_county_product_hour["pm25_product"]
    .map({"88101": 1, "88502": 2})
)

epa_selected_hourly = (
    epa_county_product_hour
    .sort_values(
        [
            "event_hour_utc",
            "County Name",
            "source_priority"
        ]
    )
    .drop_duplicates(
        subset=["event_hour_utc", "County Name"],
        keep="first"
    )
    .rename(columns={"County Name": "county"})
)

# Join with the same complete 600-row county-hour grid.
epa_hourly_sample = complete_hourly_grid.merge(
    epa_selected_hourly[
        [
            "event_hour_utc",
            "county",
            "pm25_ug_m3",
            "monitor_site_count",
            "wildfire_flagged_site_count",
            "pm25_product"
        ]
    ],
    on=["event_hour_utc", "county"],
    how="left"
)

epa_hourly_sample["pm25_available"] = (
    epa_hourly_sample["pm25_ug_m3"].notna()
)

epa_coverage_by_county = (
    epa_hourly_sample
    .groupby("county", as_index=False)
    .agg(
        hours_with_pm25=("pm25_available", "sum"),
        total_hours=("pm25_available", "size"),
        coverage_percent=(
            "pm25_available",
            lambda values: round(
                values.mean() * 100,
                2
            )
        )
    )
)

epa_interim_folder = (
    repo_root / "data" / "interim" / "epa_airdata"
)
epa_interim_folder.mkdir(
    parents=True,
    exist_ok=True
)

epa_hourly_output_path = (
    epa_interim_folder
    / "dixie_pm25_hourly_county_sample_2021-07-13_17.csv"
)

epa_hourly_sample.to_csv(
    epa_hourly_output_path,
    index=False
)

print(f"Hourly county rows: {len(epa_hourly_sample):,}")
print(
    "Rows with PM2.5 available:",
    epa_hourly_sample["pm25_available"].sum()
)
print(
    "EPA products selected:",
    epa_hourly_sample["pm25_product"]
    .value_counts(dropna=False)
    .to_dict()
)
print(
    "Saved locally to:",
    epa_hourly_output_path.relative_to(repo_root)
)

display(epa_coverage_by_county)

Hourly county rows: 600
Rows with PM2.5 available: 480
EPA products selected: {'88502': 242, '88101': 238, nan: 120}
Saved locally to: data/interim/epa_airdata/dixie_pm25_hourly_county_sample_2021-07-13_17.csv


,county,hours_with_pm25,total_hours,coverage_percent
0,Butte,120,120,100.0
1,Lassen,0,120,0.0
2,Plumas,120,120,100.0
3,Shasta,120,120,100.0
4,Tehama,120,120,100.0


### Validate PM2.5 Measurement Ranges

The hourly county measurements are summarized to identify negative values, extreme concentrations, and county-level differences. High measurements may be plausible during a wildfire and should not be removed automatically.

In [23]:
available_pm25 = epa_hourly_sample[
    epa_hourly_sample["pm25_available"]
].copy()

pm25_county_summary = (
    available_pm25
    .groupby("county", as_index=False)
    .agg(
        observed_hours=("pm25_ug_m3", "count"),
        minimum_pm25=("pm25_ug_m3", "min"),
        median_pm25=("pm25_ug_m3", "median"),
        mean_pm25=("pm25_ug_m3", "mean"),
        maximum_pm25=("pm25_ug_m3", "max")
    )
    .round(2)
)

negative_pm25_count = (
    available_pm25["pm25_ug_m3"] < 0
).sum()

wildfire_flagged_hours = (
    available_pm25[
        "wildfire_flagged_site_count"
    ].fillna(0) > 0
).sum()

print(
    f"Negative county-hour measurements: "
    f"{negative_pm25_count}"
)
print(
    f"County-hours with wildfire qualifier: "
    f"{wildfire_flagged_hours}"
)

display(pm25_county_summary)

Negative county-hour measurements: 1
County-hours with wildfire qualifier: 100


,county,observed_hours,minimum_pm25,median_pm25,mean_pm25,maximum_pm25
0,Butte,120,4.00,10.5,11.66,28.0
1,Plumas,120,-0.33,8.5,15.42,95.0
2,Shasta,120,0.00,7.0,7.80,40.0
3,Tehama,120,6.00,11.0,10.52,18.0


In [24]:
# Find the current EPA/PM2.5 DataFrame and relevant columns

for variable_name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame):
        relevant_columns = [
            column for column in value.columns
            if any(term in column.lower()
                   for term in ["pm25", "measurement", "qualifier", "parameter"])
        ]

        if relevant_columns:
            print(f"\nDataFrame: {variable_name}")
            print(f"Shape: {value.shape}")
            print("Relevant columns:", relevant_columns)


DataFrame: epa_preview
Shape: (5, 24)
Relevant columns: ['Parameter Code', 'Parameter Name', 'Sample Measurement', 'Qualifier']

DataFrame: epa_chunk
Shape: (147085, 15)
Relevant columns: ['Parameter Code', 'Sample Measurement', 'Qualifier']

DataFrame: filtered_chunk
Shape: (0, 15)
Relevant columns: ['Parameter Code', 'Sample Measurement', 'Qualifier']

DataFrame: epa_sample
Shape: (238, 15)
Relevant columns: ['Parameter Code', 'Sample Measurement', 'Qualifier']

DataFrame: epa_88502_sample
Shape: (703, 16)
Relevant columns: ['Parameter Code', 'Sample Measurement', 'Qualifier', 'pm25_product']

DataFrame: epa_88101_sample
Shape: (238, 16)
Relevant columns: ['Parameter Code', 'Sample Measurement', 'Qualifier', 'pm25_product']

DataFrame: epa_combined
Shape: (941, 19)
Relevant columns: ['Parameter Code', 'Sample Measurement', 'Qualifier', 'pm25_product', 'pm25_ug_m3']

DataFrame: epa_clean
Shape: (941, 21)
Relevant columns: ['Parameter Code', 'Sample Measurement', 'Qualifier', 'pm25_pr

In [25]:
# Show only the completed 600-row county-hour DataFrames

for variable_name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame) and len(value) == 600:
        print(f"\nDataFrame: {variable_name}")
        print(f"Shape: {value.shape}")
        print("Columns:", value.columns.tolist())


DataFrame: complete_hourly_grid
Shape: (600, 2)
Columns: ['event_hour_utc', 'county']

DataFrame: hourly_fire_sample
Shape: (600, 8)
Columns: ['event_hour_utc', 'county', 'fire_detection_count', 'total_frp_mw', 'mean_frp_mw', 'max_frp_mw', 'high_confidence_count', 'night_detection_count']

DataFrame: epa_hourly_sample
Shape: (600, 7)
Columns: ['event_hour_utc', 'county', 'pm25_ug_m3', 'monitor_site_count', 'wildfire_flagged_site_count', 'pm25_product', 'pm25_available']


In [26]:
# Preserve the original PM2.5 values and correct only negative measurements

if "pm25_ug_m3_raw" not in epa_hourly_sample.columns:
    epa_hourly_sample["pm25_ug_m3_raw"] = epa_hourly_sample["pm25_ug_m3"]

epa_hourly_sample["pm25_negative_adjusted"] = (
    epa_hourly_sample["pm25_ug_m3_raw"].notna()
    & epa_hourly_sample["pm25_ug_m3_raw"].lt(0)
)

epa_hourly_sample["pm25_ug_m3"] = (
    epa_hourly_sample["pm25_ug_m3_raw"].clip(lower=0)
)

adjusted_rows = epa_hourly_sample.loc[
    epa_hourly_sample["pm25_negative_adjusted"],
    [
        "event_hour_utc",
        "county",
        "pm25_ug_m3_raw",
        "pm25_ug_m3",
        "pm25_negative_adjusted",
    ],
]

print("Negative values adjusted:", len(adjusted_rows))
display(adjusted_rows)

Negative values adjusted: 1


,event_hour_utc,county,pm25_ug_m3_raw,pm25_ug_m3,pm25_negative_adjusted
16,2021-07-13 03:00:00+00:00,Plumas,-0.333333,0.0,True


In [27]:
# Validate the cleaned EPA county-hour dataset

print("Total county-hour rows:", len(epa_hourly_sample))
print(
    "Negative values in raw field:",
    epa_hourly_sample["pm25_ug_m3_raw"].lt(0).sum(),
)
print(
    "Negative values in cleaned field:",
    epa_hourly_sample["pm25_ug_m3"].lt(0).sum(),
)
print(
    "Flagged adjustments:",
    epa_hourly_sample["pm25_negative_adjusted"].sum(),
)
print(
    "Missing raw values:",
    epa_hourly_sample["pm25_ug_m3_raw"].isna().sum(),
)
print(
    "Missing cleaned values:",
    epa_hourly_sample["pm25_ug_m3"].isna().sum(),
)
print(
    "Maximum cleaned PM2.5:",
    epa_hourly_sample["pm25_ug_m3"].max(),
)

Total county-hour rows: 600
Negative values in raw field: 1
Negative values in cleaned field: 0
Flagged adjustments: 1
Missing raw values: 120
Missing cleaned values: 120
Maximum cleaned PM2.5: 95.0


### EPA PM2.5 Cleaning Decision

The county-hour dataset contained one negative PM2.5 value of -0.33 in Plumas County. A negative concentration is not physically meaningful and likely resulted from monitor precision or averaging near the detection limit.

The original value is preserved in `pm25_ug_m3_raw`. The cleaned `pm25_ug_m3` value is limited to a minimum of zero, and `pm25_negative_adjusted` identifies the affected row. Missing values remain missing and are not converted to zero.

The maximum value of 95.0 µg/m³ was retained because elevated PM2.5 is plausible during a wildfire period.

In [28]:
from pathlib import Path

# Locate the project root whether the notebook starts in the
# repository root or inside the notebooks folder
current_location = Path.cwd().resolve()

if current_location.name == "notebooks":
    project_root = current_location.parent
else:
    project_root = current_location

epa_output_path = (
    project_root
    / "data"
    / "interim"
    / "epa_airdata"
    / "dixie_pm25_hourly_county_sample_2021-07-13_17.csv"
)

epa_output_path.parent.mkdir(parents=True, exist_ok=True)

# Arrange the final columns in a clear order
epa_output_columns = [
    "event_hour_utc",
    "county",
    "pm25_ug_m3_raw",
    "pm25_ug_m3",
    "pm25_negative_adjusted",
    "monitor_site_count",
    "wildfire_flagged_site_count",
    "pm25_product",
    "pm25_available",
]

missing_output_columns = [
    column
    for column in epa_output_columns
    if column not in epa_hourly_sample.columns
]

assert not missing_output_columns, (
    f"Missing expected columns: {missing_output_columns}"
)

epa_hourly_sample = (
    epa_hourly_sample[epa_output_columns]
    .sort_values(["event_hour_utc", "county"])
    .reset_index(drop=True)
)

epa_hourly_sample.to_csv(epa_output_path, index=False)

print("EPA interim file saved successfully.")
print("Saved path:", epa_output_path)
print("Saved rows:", len(epa_hourly_sample))
print("Saved columns:", len(epa_hourly_sample.columns))

EPA interim file saved successfully.
Saved path: /Users/VV/Documents/GitHub/smokewatch-replay/data/interim/epa_airdata/dixie_pm25_hourly_county_sample_2021-07-13_17.csv
Saved rows: 600
Saved columns: 9


In [29]:
# Reload and validate the saved EPA interim file

epa_saved_check = pd.read_csv(
    epa_output_path,
    parse_dates=["event_hour_utc"],
)

# Convert saved flag columns back to Boolean values
for column in ["pm25_negative_adjusted", "pm25_available"]:
    epa_saved_check[column] = (
        epa_saved_check[column]
        .astype(str)
        .str.lower()
        .eq("true")
    )

print("File exists:", epa_output_path.exists())
print("Rows after reload:", len(epa_saved_check))
print("Columns after reload:", len(epa_saved_check.columns))
print(
    "Negative raw values:",
    epa_saved_check["pm25_ug_m3_raw"].lt(0).sum(),
)
print(
    "Negative cleaned values:",
    epa_saved_check["pm25_ug_m3"].lt(0).sum(),
)
print(
    "Flagged adjustments:",
    epa_saved_check["pm25_negative_adjusted"].sum(),
)
print(
    "Missing cleaned values:",
    epa_saved_check["pm25_ug_m3"].isna().sum(),
)
print(
    "Available county-hours:",
    epa_saved_check["pm25_available"].sum(),
)
print(
    "Maximum cleaned PM2.5:",
    epa_saved_check["pm25_ug_m3"].max(),
)

coverage_check = (
    epa_saved_check
    .groupby("county", as_index=False)
    .agg(
        total_hours=("event_hour_utc", "size"),
        observed_hours=("pm25_ug_m3", "count"),
        adjusted_values=("pm25_negative_adjusted", "sum"),
    )
)

coverage_check["coverage_percent"] = (
    coverage_check["observed_hours"]
    / coverage_check["total_hours"]
    * 100
).round(2)

display(coverage_check)

File exists: True
Rows after reload: 600
Columns after reload: 9
Negative raw values: 1
Negative cleaned values: 0
Flagged adjustments: 1
Missing cleaned values: 120
Available county-hours: 480
Maximum cleaned PM2.5: 95.0


,county,total_hours,observed_hours,adjusted_values,coverage_percent
0,Butte,120,120,0,100.0
1,Lassen,120,0,0,0.0
2,Plumas,120,120,1,100.0
3,Shasta,120,120,0,100.0
4,Tehama,120,120,0,100.0


### EPA AirData Feasibility Result

EPA AirData is feasible for the project when PM2.5 products 88101 and 88502 are used together. The five-day sample produced 480 observed county-hours out of the complete 600-row grid.

Butte, Plumas, Shasta, and Tehama had complete hourly coverage. Lassen had no PM2.5 observations, so its 120 county-hours remain missing rather than being filled with zero.

To avoid giving additional weight to sites with multiple instruments, measurements were first averaged to site-hour and then to county-hour. Product 88101 was preferred when both products were available. Wildfire qualifier `IT` was retained as useful source information.

One negative aggregated value was preserved in the raw field and corrected to zero only in the cleaned field with an adjustment flag. Overall, the source is suitable for the pilot, but unequal monitoring coverage must be included as a project limitation.

## NOAA GHCNh Weather Feasibility

This section evaluates NOAA Global Historical Climatology Network Hourly (GHCNh) weather observations for the five pilot counties from July 13 through July 17, 2021.

GHCNh is NOAA's current hourly weather product and replaces the older Integrated Surface Database Global Hourly format. The initial feasibility check will:

1. Identify weather stations located inside the pilot counties.
2. Confirm which stations have 2021 hourly files.
3. Download only the available station files.
4. Evaluate temperature, relative humidity, wind speed, and wind direction coverage.
5. Aggregate the observations to the same county-hour structure used for fire and PM2.5 data.

Official source:  
https://www.ncei.noaa.gov/products/global-historical-climatology-network-hourly

In [31]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import requests

# Locate the project root
current_location = Path.cwd().resolve()

if current_location.name == "notebooks":
    project_root = current_location.parent
else:
    project_root = current_location

noaa_raw_dir = project_root / "data" / "raw" / "noaa_ghcnh"
noaa_raw_dir.mkdir(parents=True, exist_ok=True)

station_catalog_url = (
    "https://www.ncei.noaa.gov/oa/"
    "global-historical-climatology-network/hourly/doc/"
    "ghcnh-station-list.csv"
)

station_catalog_path = noaa_raw_dir / "ghcnh-station-list.csv"

if not station_catalog_path.exists():
    response = requests.get(station_catalog_url, timeout=120)
    response.raise_for_status()
    station_catalog_path.write_bytes(response.content)
    print("NOAA station catalog downloaded.")
else:
    print("Using the existing NOAA station catalog.")

station_catalog = pd.read_csv(
    station_catalog_path,
    dtype={
        "GHCN_ID": "string",
        "STATE": "string",
        "ISO_CODE": "string",
    },
)

station_catalog["LATITUDE"] = pd.to_numeric(
    station_catalog["LATITUDE"],
    errors="coerce",
)

station_catalog["LONGITUDE"] = pd.to_numeric(
    station_catalog["LONGITUDE"],
    errors="coerce",
)

california_weather_stations = station_catalog.loc[
    station_catalog["STATE"].eq("CA")
    & station_catalog["ISO_CODE"].eq("US")
    & station_catalog["LATITUDE"].notna()
    & station_catalog["LONGITUDE"].notna()
].copy()

print("Total stations in catalog:", len(station_catalog))
print(
    "California stations with coordinates:",
    len(california_weather_stations),
)
print("Station catalog path:", station_catalog_path)

NOAA station catalog downloaded.
Total stations in catalog: 38870
California stations with coordinates: 391
Station catalog path: /Users/VV/Documents/GitHub/smokewatch-replay/data/raw/noaa_ghcnh/ghcnh-station-list.csv


In [32]:
# Confirm that the earlier county-boundary object is available

assert "pilot_counties" in globals(), (
    "pilot_counties was not found. "
    "Run the earlier Census county-boundary cells first."
)

pilot_counties_weather = pilot_counties.copy()

# Standardize the county-name column
if "county" in pilot_counties_weather.columns:
    county_name_column = "county"
elif "NAME" in pilot_counties_weather.columns:
    county_name_column = "NAME"
elif "NAMELSAD" in pilot_counties_weather.columns:
    county_name_column = "NAMELSAD"
else:
    raise KeyError(
        "A county-name column was not found in pilot_counties."
    )

pilot_counties_weather["county"] = (
    pilot_counties_weather[county_name_column]
    .astype(str)
    .str.replace(" County", "", regex=False)
)

pilot_counties_weather = pilot_counties_weather.loc[
    pilot_counties_weather["county"].isin(
        ["Butte", "Plumas", "Shasta", "Lassen", "Tehama"]
    )
].to_crs("EPSG:4326")

weather_station_points = gpd.GeoDataFrame(
    california_weather_stations,
    geometry=gpd.points_from_xy(
        california_weather_stations["LONGITUDE"],
        california_weather_stations["LATITUDE"],
    ),
    crs="EPSG:4326",
)

stations_in_pilot = gpd.sjoin(
    weather_station_points,
    pilot_counties_weather[["county", "geometry"]],
    how="inner",
    predicate="within",
).drop(columns=["index_right"])

stations_in_pilot = (
    stations_in_pilot
    .sort_values(["county", "NAME"])
    .reset_index(drop=True)
)

print("Stations geographically inside pilot counties:", len(stations_in_pilot))
print("Pilot counties represented:", stations_in_pilot["county"].nunique())

display(
    stations_in_pilot[
        ["GHCN_ID", "NAME", "county", "LATITUDE", "LONGITUDE"]
    ]
)

display(
    stations_in_pilot
    .groupby("county", as_index=False)
    .agg(candidate_station_count=("GHCN_ID", "nunique"))
)

Stations geographically inside pilot counties: 24
Pilot counties represented: 5


,GHCN_ID,NAME,county,LATITUDE,LONGITUDE
0,USC00041130,BRUSH CREEK RANGER STATION,Butte,39.6949,-121.3452
1,USW00093203,CHICO ARMY FLYING SCHOOL,Butte,39.8000,-121.8500
2,USC00041715,CHICO UNIVERSITY FARM,Butte,39.6911,-121.8211
3,USC00046521,OROVILLE,Butte,39.5177,-121.5530
4,USW00093210,OROVILLE MUNI AP,Butte,39.4942,-121.6222
5,USC00048544,STIRLING CITY RS,Butte,39.9041,-121.5280
6,USC00040731,BIEBER,Lassen,41.1208,-121.1347
7,USC00045623,MILFORD LAUFMAN RS,Lassen,40.1414,-120.3533
8,USC00048703,SUSANVILLE 1 WNW,Lassen,40.4239,-120.6747
9,USI0000KSVE,SUSANVILLE MUNI,Lassen,40.3757,-120.5727


,county,candidate_station_count
0,Butte,6
1,Lassen,4
2,Plumas,3
3,Shasta,7
4,Tehama,4


In [33]:
from concurrent.futures import ThreadPoolExecutor

noaa_2021_base_url = (
    "https://www.ncei.noaa.gov/oa/"
    "global-historical-climatology-network/hourly/"
    "access/by-year/2021/psv"
)

candidate_records = stations_in_pilot[
    ["GHCN_ID", "NAME", "county"]
].to_dict("records")


def check_2021_station_file(station):
    station_id = station["GHCN_ID"]
    file_url = (
        f"{noaa_2021_base_url}/"
        f"GHCNh_{station_id}_2021.psv"
    )

    try:
        response = requests.head(
            file_url,
            timeout=60,
            allow_redirects=True,
        )

        file_size_bytes = int(
            response.headers.get("content-length", 0)
        )

        return {
            **station,
            "status_code": response.status_code,
            "available_2021": response.status_code == 200,
            "file_size_mb": round(
                file_size_bytes / (1024 ** 2),
                2,
            ),
            "file_url": file_url,
        }

    except requests.RequestException as error:
        return {
            **station,
            "status_code": None,
            "available_2021": False,
            "file_size_mb": None,
            "file_url": file_url,
            "error": str(error),
        }


with ThreadPoolExecutor(max_workers=6) as executor:
    availability_records = list(
        executor.map(
            check_2021_station_file,
            candidate_records,
        )
    )

weather_station_availability = pd.DataFrame(
    availability_records
)

available_weather_stations = (
    weather_station_availability.loc[
        weather_station_availability["available_2021"]
    ]
    .sort_values(["county", "NAME"])
    .reset_index(drop=True)
)

print("Candidate stations checked:", len(weather_station_availability))
print("Stations with 2021 files:", len(available_weather_stations))
print(
    "Counties with at least one available station:",
    available_weather_stations["county"].nunique(),
)

display(
    available_weather_stations[
        [
            "GHCN_ID",
            "NAME",
            "county",
            "file_size_mb",
            "status_code",
        ]
    ]
)

display(
    available_weather_stations
    .groupby("county", as_index=False)
    .agg(available_station_count=("GHCN_ID", "nunique"))
)

Candidate stations checked: 24
Stations with 2021 files: 23
Counties with at least one available station: 5


,GHCN_ID,NAME,county,file_size_mb,status_code
0,USC00041130,BRUSH CREEK RANGER STATION,Butte,13.90,200
1,USW00093203,CHICO ARMY FLYING SCHOOL,Butte,12.30,200
2,USC00041715,CHICO UNIVERSITY FARM,Butte,13.86,200
3,USC00046521,OROVILLE,Butte,13.51,200
4,USW00093210,OROVILLE MUNI AP,Butte,9.05,200
5,USC00048544,STIRLING CITY RS,Butte,12.29,200
6,USC00040731,BIEBER,Lassen,14.75,200
7,USC00045623,MILFORD LAUFMAN RS,Lassen,15.07,200
8,USC00048703,SUSANVILLE 1 WNW,Lassen,14.96,200
9,USI0000KSVE,SUSANVILLE MUNI,Lassen,17.95,200


,county,available_station_count
0,Butte,6
1,Lassen,4
2,Plumas,3
3,Shasta,6
4,Tehama,4


### NOAA Station Selection Method

The station catalog contained 24 stations inside the pilot counties, and 23 had a 2021 hourly file. Downloading every annual file would add unnecessary volume and would give more weight to counties with more stations.

For the feasibility sample, one representative station is selected for each county. Airport or municipal stations are preferred because they generally provide frequent temperature, humidity, and wind observations.

The first selection is:

- Butte: Oroville Municipal Airport
- Lassen: Susanville Municipal Airport
- Plumas: Portola
- Shasta: Redding Airport
- Tehama: Red Bluff Municipal Airport

The five-day field coverage will be checked before these stations are accepted. If a station lacks the required weather variables, a documented nearby proxy will be evaluated.

In [34]:
# Select one representative station per pilot county

selected_weather_stations = pd.DataFrame(
    [
        {
            "county": "Butte",
            "GHCN_ID": "USW00093210",
            "station_name": "OROVILLE MUNI AP",
        },
        {
            "county": "Lassen",
            "GHCN_ID": "USI0000KSVE",
            "station_name": "SUSANVILLE MUNI",
        },
        {
            "county": "Plumas",
            "GHCN_ID": "USC00047085",
            "station_name": "PORTOLA",
        },
        {
            "county": "Shasta",
            "GHCN_ID": "USW00024257",
            "station_name": "REDDING AP",
        },
        {
            "county": "Tehama",
            "GHCN_ID": "USW00024216",
            "station_name": "RED BLUFF MUNI AP",
        },
    ]
)

selected_weather_stations["file_url"] = (
    noaa_2021_base_url
    + "/GHCNh_"
    + selected_weather_stations["GHCN_ID"]
    + "_2021.psv"
)

display(selected_weather_stations)

,county,GHCN_ID,station_name,file_url
0,Butte,USW00093210,OROVILLE MUNI AP,https://www.ncei.noaa.gov/oa/global-historical...
1,Lassen,USI0000KSVE,SUSANVILLE MUNI,https://www.ncei.noaa.gov/oa/global-historical...
2,Plumas,USC00047085,PORTOLA,https://www.ncei.noaa.gov/oa/global-historical...
3,Shasta,USW00024257,REDDING AP,https://www.ncei.noaa.gov/oa/global-historical...
4,Tehama,USW00024216,RED BLUFF MUNI AP,https://www.ncei.noaa.gov/oa/global-historical...


In [35]:
# Download the selected 2021 station files

download_results = []

for station in selected_weather_stations.to_dict("records"):
    station_id = station["GHCN_ID"]
    station_path = (
        noaa_raw_dir
        / f"GHCNh_{station_id}_2021.psv"
    )

    if not station_path.exists():
        with requests.get(
            station["file_url"],
            stream=True,
            timeout=(30, 300),
        ) as response:
            response.raise_for_status()

            with station_path.open("wb") as file:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        file.write(chunk)

        download_status = "downloaded"
    else:
        download_status = "existing file"

    download_results.append(
        {
            "county": station["county"],
            "GHCN_ID": station_id,
            "station_name": station["station_name"],
            "status": download_status,
            "file_size_mb": round(
                station_path.stat().st_size / (1024 ** 2),
                2,
            ),
            "local_path": str(station_path),
        }
    )

weather_download_summary = pd.DataFrame(download_results)

print("Station files ready:", len(weather_download_summary))
print(
    "Combined file size:",
    round(weather_download_summary["file_size_mb"].sum(), 2),
    "MB",
)

display(
    weather_download_summary[
        [
            "county",
            "station_name",
            "status",
            "file_size_mb",
        ]
    ]
)

Station files ready: 5
Combined file size: 60.28 MB


,county,station_name,status,file_size_mb
0,Butte,OROVILLE MUNI AP,downloaded,9.05
1,Lassen,SUSANVILLE MUNI,downloaded,17.95
2,Plumas,PORTOLA,downloaded,12.72
3,Shasta,REDDING AP,downloaded,10.55
4,Tehama,RED BLUFF MUNI AP,downloaded,10.01


In [36]:
# Read the five-day weather sample from each selected station

weather_columns = [
    "STATION",
    "Station_name",
    "DATE",
    "LATITUDE",
    "LONGITUDE",
    "ELEVATION",
    "temperature",
    "temperature_Quality_Code",
    "relative_humidity",
    "relative_humidity_Quality_Code",
    "wind_speed",
    "wind_speed_Quality_Code",
    "wind_direction",
    "wind_direction_Quality_Code",
]

sample_start_utc = pd.Timestamp(
    "2021-07-13 00:00:00",
    tz="UTC",
)

sample_end_utc = pd.Timestamp(
    "2021-07-18 00:00:00",
    tz="UTC",
)

weather_sample_parts = []

for station in selected_weather_stations.to_dict("records"):
    station_path = (
        noaa_raw_dir
        / f"GHCNh_{station['GHCN_ID']}_2021.psv"
    )

    station_year = pd.read_csv(
        station_path,
        sep="|",
        usecols=weather_columns,
        low_memory=False,
    )

    station_year["event_timestamp_utc"] = pd.to_datetime(
        station_year["DATE"],
        errors="coerce",
        utc=True,
    )

    station_sample = station_year.loc[
        station_year["event_timestamp_utc"].ge(sample_start_utc)
        & station_year["event_timestamp_utc"].lt(sample_end_utc)
    ].copy()

    station_sample["county"] = station["county"]

    weather_sample_parts.append(station_sample)

weather_raw_sample = pd.concat(
    weather_sample_parts,
    ignore_index=True,
)

weather_field_coverage = (
    weather_raw_sample
    .groupby(
        ["county", "STATION", "Station_name"],
        as_index=False,
    )
    .agg(
        raw_records=("event_timestamp_utc", "size"),
        temperature_records=("temperature", "count"),
        humidity_records=("relative_humidity", "count"),
        wind_speed_records=("wind_speed", "count"),
        wind_direction_records=("wind_direction", "count"),
    )
)

print("Five-day weather records:", len(weather_raw_sample))
print(
    "Invalid timestamps:",
    weather_raw_sample["event_timestamp_utc"].isna().sum(),
)
print(
    "Date range:",
    weather_raw_sample["event_timestamp_utc"].min(),
    "to",
    weather_raw_sample["event_timestamp_utc"].max(),
)

display(weather_field_coverage)

Five-day weather records: 1240
Invalid timestamps: 0
Date range: 2021-07-13 00:00:00+00:00 to 2021-07-17 23:55:00+00:00


,county,STATION,Station_name,raw_records,temperature_records,humidity_records,wind_speed_records,wind_direction_records
0,Butte,USW00093210,OROVILLE MUNI AP,120,120,120,119,119
1,Lassen,USI0000KSVE,SUSANVILLE MUNI,360,360,360,360,360
2,Plumas,USC00047085,PORTOLA,480,0,0,0,0
3,Shasta,USW00024257,REDDING AP,140,140,140,139,139
4,Tehama,USW00024216,RED BLUFF MUNI AP,140,140,140,140,140


### Plumas Weather Coverage Check

The Portola station had 480 records during the five-day period but did not contain usable temperature, relative-humidity, wind-speed, or wind-direction measurements.

Therefore, file availability alone is not sufficient for station selection. Portola remains in the raw-data folder as evidence, but it will not be used in the cleaned weather dataset.

For Plumas, the geographically nearest station among the already validated stations with all four required weather fields will be used as a proxy. The proxy station ID and assignment will remain visible in the final dataset.

In [37]:
# Find the nearest validated weather station to Plumas County

required_weather_fields = [
    "temperature_records",
    "humidity_records",
    "wind_speed_records",
    "wind_direction_records",
]

validated_station_ids = weather_field_coverage.loc[
    weather_field_coverage[required_weather_fields]
    .gt(0)
    .all(axis=1),
    "STATION",
].tolist()

proxy_candidate_metadata = station_catalog.loc[
    station_catalog["GHCN_ID"].isin(validated_station_ids),
    [
        "GHCN_ID",
        "NAME",
        "LATITUDE",
        "LONGITUDE",
        "ELEVATION",
    ],
].copy()

proxy_candidate_points = gpd.GeoDataFrame(
    proxy_candidate_metadata,
    geometry=gpd.points_from_xy(
        proxy_candidate_metadata["LONGITUDE"],
        proxy_candidate_metadata["LATITUDE"],
    ),
    crs="EPSG:4326",
).to_crs("EPSG:3310")

plumas_geometry = (
    pilot_counties_weather.loc[
        pilot_counties_weather["county"].eq("Plumas")
    ]
    .to_crs("EPSG:3310")
    .geometry
    .union_all()
)

proxy_candidate_points["distance_to_plumas_km"] = (
    proxy_candidate_points.geometry.distance(plumas_geometry)
    / 1000
)

proxy_candidate_points = (
    proxy_candidate_points
    .sort_values("distance_to_plumas_km")
    .reset_index(drop=True)
)

plumas_proxy_choice = proxy_candidate_points.iloc[0]
plumas_proxy_id = plumas_proxy_choice["GHCN_ID"]

print("Selected Plumas proxy station:")
print("Station ID:", plumas_proxy_id)
print("Station name:", plumas_proxy_choice["NAME"])
print(
    "Distance from Plumas County boundary:",
    round(plumas_proxy_choice["distance_to_plumas_km"], 2),
    "km",
)

display(
    proxy_candidate_points[
        [
            "GHCN_ID",
            "NAME",
            "ELEVATION",
            "distance_to_plumas_km",
        ]
    ]
)

Selected Plumas proxy station:
Station ID: USI0000KSVE
Station name: SUSANVILLE MUNI
Distance from Plumas County boundary: 10.03 km


,GHCN_ID,NAME,ELEVATION,distance_to_plumas_km
0,USI0000KSVE,SUSANVILLE MUNI,1264.6,10.031496
1,USW00093210,OROVILLE MUNI AP,57.0,41.899207
2,USW00024257,REDDING AP,152.7,68.279245
3,USW00024216,RED BLUFF MUNI AP,107.9,68.863061


In [38]:
# Build the five-county weather sample

in_county_weather = weather_raw_sample.loc[
    weather_raw_sample["county"].ne("Plumas")
].copy()

in_county_weather["station_assignment"] = "in_county"

plumas_proxy_weather = weather_raw_sample.loc[
    weather_raw_sample["STATION"].eq(plumas_proxy_id)
].copy()

plumas_proxy_weather["county"] = "Plumas"
plumas_proxy_weather["station_assignment"] = "nearby_proxy"

weather_analysis_sample = pd.concat(
    [
        in_county_weather,
        plumas_proxy_weather,
    ],
    ignore_index=True,
)

weather_assignment_check = (
    weather_analysis_sample
    .groupby(
        [
            "county",
            "STATION",
            "Station_name",
            "station_assignment",
        ],
        as_index=False,
    )
    .agg(raw_records=("event_timestamp_utc", "size"))
)

print(
    "Weather records prepared for cleaning:",
    len(weather_analysis_sample),
)
print(
    "Counties represented:",
    weather_analysis_sample["county"].nunique(),
)

display(weather_assignment_check)

Weather records prepared for cleaning: 1120
Counties represented: 5


,county,STATION,Station_name,station_assignment,raw_records
0,Butte,USW00093210,OROVILLE MUNI AP,in_county,120
1,Lassen,USI0000KSVE,SUSANVILLE MUNI,in_county,360
2,Plumas,USI0000KSVE,SUSANVILLE MUNI,nearby_proxy,360
3,Shasta,USW00024257,REDDING AP,in_county,140
4,Tehama,USW00024216,RED BLUFF MUNI AP,in_county,140


In [43]:
# Clean weather measurements while preserving NOAA special codes

accepted_quality_codes = {
    "",
    "0",
    "1",
    "4",
    "5",
    "9",
    "A",
    "U",
    "P",
    "I",
    "M",
    "R",
}

weather_variable_rules = {
    "temperature": {
        "clean_column": "temperature_c",
        "minimum": -60,
        "maximum": 60,
    },
    "relative_humidity": {
        "clean_column": "relative_humidity_percent",
        "minimum": 0,
        "maximum": 100,
    },
    "wind_speed": {
        "clean_column": "wind_speed_m_s",
        "minimum": 0,
        "maximum": 100,
    },
    "wind_direction": {
        "clean_column": "wind_direction_degrees",
        "minimum": 0,
        "maximum": 360,
    },
}

quality_results = []

for source_column, rule in weather_variable_rules.items():
    clean_column = rule["clean_column"]
    quality_column = f"{source_column}_Quality_Code"

    raw_values = pd.to_numeric(
        weather_analysis_sample[source_column],
        errors="coerce",
    )

    weather_analysis_sample[clean_column] = raw_values.copy()

    normalized_quality_codes = (
        weather_analysis_sample[quality_column]
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .fillna("")
    )

    quality_valid = normalized_quality_codes.isin(
        accepted_quality_codes
    )

    rejected_by_quality = (
        raw_values.notna()
        & ~quality_valid
    )

    # NOAA uses 999 when a directional angle is unavailable.
    if source_column == "wind_direction":
        special_code_mask = raw_values.eq(999)

        measurement_type = pd.Series(
    np.where(
        special_code_mask,
        "NOAA_999_unavailable_or_variable",
        "",
    ),
    index=weather_analysis_sample.index,
        )

        weather_analysis_sample[
            "wind_direction_special_code"
        ] = special_code_mask

        weather_analysis_sample[
            "wind_direction_measurement_type"
        ] = measurement_type
    else:
        special_code_mask = pd.Series(
            False,
            index=weather_analysis_sample.index,
        )

    outside_physical_range = (
        raw_values.notna()
        & ~special_code_mask
        & ~raw_values.between(
            rule["minimum"],
            rule["maximum"],
        )
    )

    weather_analysis_sample.loc[
        rejected_by_quality
        | special_code_mask
        | outside_physical_range,
        clean_column,
    ] = np.nan

    quality_results.append(
        {
            "variable": source_column,
            "quality_rejections": int(
                rejected_by_quality.sum()
            ),
            "special_code_values": int(
                special_code_mask.sum()
            ),
            "physical_range_rejections": int(
                outside_physical_range.sum()
            ),
            "remaining_values": int(
                weather_analysis_sample[
                    clean_column
                ].notna().sum()
            ),
        }
    )

weather_quality_summary = pd.DataFrame(quality_results)

special_direction_breakdown = (
    weather_analysis_sample.loc[
        weather_analysis_sample[
            "wind_direction_special_code"
        ],
        "wind_direction_measurement_type",
    ]
    .value_counts(dropna=False)
    .rename_axis("measurement_type")
    .reset_index(name="record_count")
)

display(weather_quality_summary)

print("Wind-direction special-code breakdown:")
display(special_direction_breakdown)

,variable,quality_rejections,special_code_values,physical_range_rejections,remaining_values
0,temperature,0,0,0,1120
1,relative_humidity,0,0,0,1120
2,wind_speed,0,0,0,1118
3,wind_direction,0,199,0,919


Wind-direction special-code breakdown:


,measurement_type,record_count
0,NOAA_999_unavailable_or_variable,199


In [45]:
# Calculate a circular mean for wind direction

def circular_mean_degrees(values):
    valid_values = pd.to_numeric(
        pd.Series(values),
        errors="coerce",
    ).dropna()

    if valid_values.empty:
        return np.nan

    radians = np.deg2rad(valid_values)

    mean_sine = np.sin(radians).mean()
    mean_cosine = np.cos(radians).mean()

    if np.isclose(mean_sine, 0) and np.isclose(mean_cosine, 0):
        return np.nan

    return float(
        np.rad2deg(
            np.arctan2(mean_sine, mean_cosine)
        ) % 360
    )


weather_analysis_sample["event_hour_utc"] = (
    weather_analysis_sample["event_timestamp_utc"]
    .dt.floor("h")
)

weather_station_hourly = (
    weather_analysis_sample
    .groupby(
        [
            "event_hour_utc",
            "county",
            "STATION",
            "Station_name",
            "station_assignment",
        ],
        as_index=False,
    )
    .agg(
        weather_observation_count=(
            "event_timestamp_utc",
            "size",
        ),
        temperature_c=(
            "temperature_c",
            "mean",
        ),
        relative_humidity_percent=(
            "relative_humidity_percent",
            "mean",
        ),
        wind_speed_m_s=(
            "wind_speed_m_s",
            "mean",
        ),
        maximum_wind_speed_m_s=(
            "wind_speed_m_s",
            "max",
        ),
        wind_direction_degrees=(
            "wind_direction_degrees",
            circular_mean_degrees,
        ),
    )
)

weather_hourly_sample = complete_hourly_grid.merge(
    weather_station_hourly,
    on=["event_hour_utc", "county"],
    how="left",
    validate="one_to_one",
)

weather_hourly_sample["weather_available"] = (
    weather_hourly_sample[
        [
            "temperature_c",
            "relative_humidity_percent",
            "wind_speed_m_s",
        ]
    ]
    .notna()
    .any(axis=1)
)

weather_hourly_sample["weather_proxy_used"] = (
    weather_hourly_sample["station_assignment"]
    .eq("nearby_proxy")
)

weather_coverage_summary = (
    weather_hourly_sample
    .groupby("county", as_index=False)
    .agg(
        total_hours=("event_hour_utc", "size"),
        available_hours=("weather_available", "sum"),
        temperature_hours=("temperature_c", "count"),
        humidity_hours=(
            "relative_humidity_percent",
            "count",
        ),
        wind_speed_hours=("wind_speed_m_s", "count"),
        wind_direction_hours=(
            "wind_direction_degrees",
            "count",
        ),
        proxy_hours=("weather_proxy_used", "sum"),
    )
)

weather_coverage_summary["coverage_percent"] = (
    weather_coverage_summary["available_hours"]
    / weather_coverage_summary["total_hours"]
    * 100
).round(2)

print("Final county-hour rows:", len(weather_hourly_sample))
print(
    "County-hours with weather:",
    weather_hourly_sample["weather_available"].sum(),
)

display(weather_coverage_summary)

Final county-hour rows: 600
County-hours with weather: 600


,county,total_hours,available_hours,temperature_hours,humidity_hours,wind_speed_hours,wind_direction_hours,proxy_hours,coverage_percent
0,Butte,120,120,120,120,119,103,0,100.0
1,Lassen,120,120,120,120,120,115,0,100.0
2,Plumas,120,120,120,120,120,115,120,100.0
3,Shasta,120,120,120,120,119,67,0,100.0
4,Tehama,120,120,120,120,120,91,0,100.0


In [46]:
# Preserve the hourly count of NOAA wind-direction special codes

wind_special_hourly = (
    weather_analysis_sample
    .groupby(
        ["event_hour_utc", "county"],
        as_index=False,
    )
    .agg(
        wind_direction_special_count=(
            "wind_direction_special_code",
            "sum",
        )
    )
)

weather_hourly_sample = (
    weather_hourly_sample
    .drop(
        columns=["wind_direction_special_count"],
        errors="ignore",
    )
    .merge(
        wind_special_hourly,
        on=["event_hour_utc", "county"],
        how="left",
        validate="one_to_one",
    )
)

weather_hourly_sample[
    "wind_direction_special_count"
] = (
    weather_hourly_sample[
        "wind_direction_special_count"
    ]
    .fillna(0)
    .astype(int)
)

weather_range_summary = (
    weather_hourly_sample
    .groupby("county", as_index=False)
    .agg(
        temperature_min_c=("temperature_c", "min"),
        temperature_mean_c=("temperature_c", "mean"),
        temperature_max_c=("temperature_c", "max"),
        humidity_min_percent=(
            "relative_humidity_percent",
            "min",
        ),
        humidity_mean_percent=(
            "relative_humidity_percent",
            "mean",
        ),
        humidity_max_percent=(
            "relative_humidity_percent",
            "max",
        ),
        wind_speed_mean_m_s=("wind_speed_m_s", "mean"),
        wind_speed_max_m_s=(
            "maximum_wind_speed_m_s",
            "max",
        ),
    )
    .round(2)
)

# Structural and physical-range checks
assert len(weather_hourly_sample) == 600
assert not weather_hourly_sample[
    ["event_hour_utc", "county"]
].duplicated().any()

assert weather_hourly_sample[
    "temperature_c"
].dropna().between(-60, 60).all()

assert weather_hourly_sample[
    "relative_humidity_percent"
].dropna().between(0, 100).all()

assert weather_hourly_sample[
    "wind_speed_m_s"
].dropna().between(0, 100).all()

assert weather_hourly_sample[
    "wind_direction_degrees"
].dropna().between(0, 360).all()

print("Final rows:", len(weather_hourly_sample))
print(
    "Duplicate county-hour keys:",
    weather_hourly_sample[
        ["event_hour_utc", "county"]
    ].duplicated().sum(),
)
print(
    "County-hours with weather:",
    weather_hourly_sample["weather_available"].sum(),
)
print(
    "County-hours using a proxy:",
    weather_hourly_sample["weather_proxy_used"].sum(),
)
print(
    "Missing temperature hours:",
    weather_hourly_sample["temperature_c"].isna().sum(),
)
print(
    "Missing humidity hours:",
    weather_hourly_sample[
        "relative_humidity_percent"
    ].isna().sum(),
)
print(
    "Missing wind-speed hours:",
    weather_hourly_sample["wind_speed_m_s"].isna().sum(),
)
print(
    "Missing wind-direction hours:",
    weather_hourly_sample[
        "wind_direction_degrees"
    ].isna().sum(),
)
print(
    "Raw NOAA 999 reports preserved:",
    weather_hourly_sample[
        "wind_direction_special_count"
    ].sum(),
)
print(
    "County-hours containing a NOAA 999 report:",
    weather_hourly_sample[
        "wind_direction_special_count"
    ].gt(0).sum(),
)

display(weather_range_summary)

Final rows: 600
Duplicate county-hour keys: 0
County-hours with weather: 600
County-hours using a proxy: 120
Missing temperature hours: 0
Missing humidity hours: 0
Missing wind-speed hours: 2
Missing wind-direction hours: 109
Raw NOAA 999 reports preserved: 199
County-hours containing a NOAA 999 report: 156


,county,temperature_min_c,temperature_mean_c,temperature_max_c,humidity_min_percent,humidity_mean_percent,humidity_max_percent,wind_speed_mean_m_s,wind_speed_max_m_s
0,Butte,16.10,26.62,37.80,11.00,40.45,70.0,2.97,7.2
1,Lassen,10.00,24.87,37.67,3.33,19.81,47.0,3.53,9.8
2,Plumas,10.00,24.87,37.67,3.33,19.81,47.0,3.53,9.8
3,Shasta,18.90,28.89,40.60,7.00,29.78,63.0,2.31,8.2
4,Tehama,18.85,28.07,39.40,11.50,36.81,64.0,2.98,8.8


In [47]:
# Save the final NOAA county-hour sample

weather_output_path = (
    project_root
    / "data"
    / "interim"
    / "noaa_ghcnh"
    / "dixie_weather_hourly_county_sample_2021-07-13_17.csv"
)

weather_output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

weather_output = weather_hourly_sample.rename(
    columns={
        "STATION": "weather_station_id",
        "Station_name": "weather_station_name",
    }
).copy()

weather_output_columns = [
    "event_hour_utc",
    "county",
    "weather_station_id",
    "weather_station_name",
    "station_assignment",
    "weather_observation_count",
    "temperature_c",
    "relative_humidity_percent",
    "wind_speed_m_s",
    "maximum_wind_speed_m_s",
    "wind_direction_degrees",
    "wind_direction_special_count",
    "weather_available",
    "weather_proxy_used",
]

missing_output_columns = [
    column
    for column in weather_output_columns
    if column not in weather_output.columns
]

assert not missing_output_columns, (
    f"Missing expected columns: {missing_output_columns}"
)

weather_output = (
    weather_output[weather_output_columns]
    .sort_values(["event_hour_utc", "county"])
    .reset_index(drop=True)
)

weather_output.to_csv(
    weather_output_path,
    index=False,
)

weather_saved_check = pd.read_csv(
    weather_output_path,
    parse_dates=["event_hour_utc"],
)

for column in [
    "weather_available",
    "weather_proxy_used",
]:
    weather_saved_check[column] = (
        weather_saved_check[column]
        .astype(str)
        .str.lower()
        .eq("true")
    )

print("Weather interim file saved successfully.")
print("Saved path:", weather_output_path)
print("Rows after reload:", len(weather_saved_check))
print("Columns after reload:", len(weather_saved_check.columns))
print(
    "Duplicate county-hour keys:",
    weather_saved_check[
        ["event_hour_utc", "county"]
    ].duplicated().sum(),
)
print(
    "Available county-hours:",
    weather_saved_check["weather_available"].sum(),
)
print(
    "Proxy county-hours:",
    weather_saved_check["weather_proxy_used"].sum(),
)
print(
    "NOAA 999 reports:",
    weather_saved_check[
        "wind_direction_special_count"
    ].sum(),
)

Weather interim file saved successfully.
Saved path: /Users/VV/Documents/GitHub/smokewatch-replay/data/interim/noaa_ghcnh/dixie_weather_hourly_county_sample_2021-07-13_17.csv
Rows after reload: 600
Columns after reload: 14
Duplicate county-hour keys: 0
Available county-hours: 600
Proxy county-hours: 120
NOAA 999 reports: 199


### NOAA GHCNh Feasibility Result

NOAA GHCNh is feasible for the weather component of the project. Temperature and relative humidity were available for all 600 county-hours. Wind speed was available for 598 county-hours.

One representative station was used for each county to avoid giving more weight to counties with more stations. The Portola station had timestamped records but did not provide the required temperature, humidity, or wind variables. Therefore, Susanville Municipal Airport was used as a weather proxy for Plumas County. The station is approximately 10.03 km from the Plumas County boundary, and all 120 proxy hours are explicitly flagged.

NOAA value `999` was treated as a special unavailable or variable wind-direction code rather than a physical outlier. The raw occurrence count was preserved, while only valid directional angles were used to calculate circular hourly means.

The source is suitable for the pilot, but the Plumas proxy and reduced wind-direction coverage must remain documented as limitations.

## Three-Source County-Hour Integration

This section combines the validated NASA fire, EPA PM2.5, and NOAA weather samples using `event_hour_utc` and `county`.

The integration will use the saved interim files rather than relying only on notebook memory. This confirms that the persisted outputs can be reloaded and joined reproducibly.

The validation will distinguish:

- Structural join completeness
- Fire-detection presence
- PM2.5 measurement availability
- Weather measurement availability
- Hours where all required measurements are available

In [48]:
# Load the three saved interim datasets

from pathlib import Path
import pandas as pd

current_location = Path.cwd().resolve()

if current_location.name == "notebooks":
    project_root = current_location.parent
else:
    project_root = current_location

fire_input_path = (
    project_root
    / "data"
    / "interim"
    / "nasa_firms"
    / "dixie_fire_hourly_county_sample_2021-07-13_17.csv"
)

pm25_input_path = (
    project_root
    / "data"
    / "interim"
    / "epa_airdata"
    / "dixie_pm25_hourly_county_sample_2021-07-13_17.csv"
)

weather_input_path = (
    project_root
    / "data"
    / "interim"
    / "noaa_ghcnh"
    / "dixie_weather_hourly_county_sample_2021-07-13_17.csv"
)

input_paths = {
    "NASA fire": fire_input_path,
    "EPA PM2.5": pm25_input_path,
    "NOAA weather": weather_input_path,
}

missing_files = [
    f"{name}: {path}"
    for name, path in input_paths.items()
    if not path.exists()
]

assert not missing_files, (
    "Missing required interim files:\n"
    + "\n".join(missing_files)
)

fire_integration_input = pd.read_csv(fire_input_path)
pm25_integration_input = pd.read_csv(pm25_input_path)
weather_integration_input = pd.read_csv(weather_input_path)

for dataframe in [
    fire_integration_input,
    pm25_integration_input,
    weather_integration_input,
]:
    dataframe["event_hour_utc"] = pd.to_datetime(
        dataframe["event_hour_utc"],
        errors="coerce",
        utc=True,
    )

print("All three interim files loaded successfully.")
print("NASA rows:", len(fire_integration_input))
print("EPA rows:", len(pm25_integration_input))
print("NOAA rows:", len(weather_integration_input))

All three interim files loaded successfully.
NASA rows: 600
EPA rows: 600
NOAA rows: 600


In [50]:
# Validate the structure of each persisted dataset

def validate_county_hour_source(source_name, dataframe):
    return {
        "source": source_name,
        "rows": len(dataframe),
        "columns": len(dataframe.columns),
        "invalid_timestamps": int(
            dataframe["event_hour_utc"].isna().sum()
        ),
        "duplicate_keys": int(
            dataframe[
                ["event_hour_utc", "county"]
            ].duplicated().sum()
        ),
        "counties": int(dataframe["county"].nunique()),
        "unique_hours": int(
            dataframe["event_hour_utc"].nunique()
        ),
        "earliest_hour": dataframe[
            "event_hour_utc"
        ].min(),
        "latest_hour": dataframe[
            "event_hour_utc"
        ].max(),
    }


source_structure_check = pd.DataFrame(
    [
        validate_county_hour_source(
            "NASA fire",
            fire_integration_input,
        ),
        validate_county_hour_source(
            "EPA PM2.5",
            pm25_integration_input,
        ),
        validate_county_hour_source(
            "NOAA weather",
            weather_integration_input,
        ),
    ]
)

assert source_structure_check["rows"].eq(600).all()
assert source_structure_check["invalid_timestamps"].eq(0).all()
assert source_structure_check["duplicate_keys"].eq(0).all()
assert source_structure_check["counties"].eq(5).all()
assert source_structure_check["unique_hours"].eq(120).all()

display(source_structure_check)

,source,rows,columns,invalid_timestamps,duplicate_keys,counties,unique_hours,earliest_hour,latest_hour
0,NASA fire,600,8,0,0,5,120,2021-07-13 00:00:00+00:00,2021-07-17 23:00:00+00:00
1,EPA PM2.5,600,9,0,0,5,120,2021-07-13 00:00:00+00:00,2021-07-17 23:00:00+00:00
2,NOAA weather,600,14,0,0,5,120,2021-07-13 00:00:00+00:00,2021-07-17 23:00:00+00:00


In [51]:
# Convert persisted flag columns to Boolean values

def normalize_boolean_column(dataframe, column):
    if column not in dataframe.columns:
        return

    dataframe[column] = (
        dataframe[column]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("true")
    )


for column in [
    "pm25_available",
    "pm25_negative_adjusted",
]:
    normalize_boolean_column(
        pm25_integration_input,
        column,
    )

for column in [
    "weather_available",
    "weather_proxy_used",
]:
    normalize_boolean_column(
        weather_integration_input,
        column,
    )

# Add source-row markers before joining
fire_for_join = fire_integration_input.assign(
    fire_row_present=True
)

pm25_for_join = pm25_integration_input.assign(
    pm25_row_present=True
)

weather_for_join = weather_integration_input.assign(
    weather_row_present=True
)

integrated_sample = (
    fire_for_join
    .merge(
        pm25_for_join,
        on=["event_hour_utc", "county"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        weather_for_join,
        on=["event_hour_utc", "county"],
        how="left",
        validate="one_to_one",
    )
)

for column in [
    "fire_row_present",
    "pm25_row_present",
    "weather_row_present",
]:
    integrated_sample[column] = (
        integrated_sample[column]
        .fillna(False)
        .astype(bool)
    )

integrated_sample["fire_detection_present"] = (
    integrated_sample["fire_detection_count"].gt(0)
)

integrated_sample["all_measurements_available"] = (
    integrated_sample["pm25_available"]
    & integrated_sample["weather_available"]
)

print("Integrated rows:", len(integrated_sample))
print(
    "Duplicate county-hour keys:",
    integrated_sample[
        ["event_hour_utc", "county"]
    ].duplicated().sum(),
)
print(
    "NASA structural rows:",
    integrated_sample["fire_row_present"].sum(),
)
print(
    "EPA structural rows:",
    integrated_sample["pm25_row_present"].sum(),
)
print(
    "NOAA structural rows:",
    integrated_sample["weather_row_present"].sum(),
)
print(
    "County-hours with fire detections:",
    integrated_sample["fire_detection_present"].sum(),
)
print(
    "County-hours with PM2.5:",
    integrated_sample["pm25_available"].sum(),
)
print(
    "County-hours with weather:",
    integrated_sample["weather_available"].sum(),
)
print(
    "County-hours with all measurements:",
    integrated_sample[
        "all_measurements_available"
    ].sum(),
)

Integrated rows: 600
Duplicate county-hour keys: 0
NASA structural rows: 600
EPA structural rows: 600
NOAA structural rows: 600
County-hours with fire detections: 19
County-hours with PM2.5: 480
County-hours with weather: 600
County-hours with all measurements: 480


In [52]:
# Summarize integration and measurement coverage by county

integration_coverage_summary = (
    integrated_sample
    .groupby("county", as_index=False)
    .agg(
        total_hours=("event_hour_utc", "size"),
        fire_structural_rows=("fire_row_present", "sum"),
        pm25_structural_rows=("pm25_row_present", "sum"),
        weather_structural_rows=(
            "weather_row_present",
            "sum",
        ),
        fire_detection_hours=(
            "fire_detection_present",
            "sum",
        ),
        pm25_available_hours=(
            "pm25_available",
            "sum",
        ),
        weather_available_hours=(
            "weather_available",
            "sum",
        ),
        all_measurements_available_hours=(
            "all_measurements_available",
            "sum",
        ),
        weather_proxy_hours=(
            "weather_proxy_used",
            "sum",
        ),
        pm25_adjusted_hours=(
            "pm25_negative_adjusted",
            "sum",
        ),
    )
)

integration_coverage_summary[
    "all_measurements_coverage_percent"
] = (
    integration_coverage_summary[
        "all_measurements_available_hours"
    ]
    / integration_coverage_summary["total_hours"]
    * 100
).round(2)

display(integration_coverage_summary)

,county,total_hours,fire_structural_rows,pm25_structural_rows,weather_structural_rows,fire_detection_hours,pm25_available_hours,weather_available_hours,all_measurements_available_hours,weather_proxy_hours,pm25_adjusted_hours,all_measurements_coverage_percent
0,Butte,120,120,120,120,2,120,120,120,0,0,100.0
1,Lassen,120,120,120,120,2,0,120,0,0,0,0.0
2,Plumas,120,120,120,120,15,120,120,120,120,1,100.0
3,Shasta,120,120,120,120,0,120,120,120,0,0,100.0
4,Tehama,120,120,120,120,0,120,120,120,0,0,100.0


## Integrated Dataset Validation and Persistence

The joined county-hour dataset is now validated against the source datasets. These checks confirm that aggregation totals, missingness, quality flags, proxy assignments, and structural coverage survived the three-source join.

In [53]:
validation_results = pd.DataFrame(
    [
        ("Integrated county-hour rows", len(integrated_sample), 600),
        (
            "Duplicate county-hour keys",
            integrated_sample.duplicated(
                subset=["event_hour_utc", "county"]
            ).sum(),
            0,
        ),
        ("NASA structural rows", integrated_sample["fire_row_present"].sum(), 600),
        ("EPA structural rows", integrated_sample["pm25_row_present"].sum(), 600),
        ("NOAA structural rows", integrated_sample["weather_row_present"].sum(), 600),
        (
            "Total fire detections preserved",
            integrated_sample["fire_detection_count"].sum(),
            976,
        ),
        (
            "County-hours with fire detections",
            integrated_sample["fire_detection_present"].sum(),
            19,
        ),
        (
            "County-hours with PM2.5",
            integrated_sample["pm25_available"].sum(),
            480,
        ),
        (
            "County-hours without PM2.5",
            (~integrated_sample["pm25_available"]).sum(),
            120,
        ),
        (
            "Negative PM2.5 values adjusted",
            integrated_sample["pm25_negative_adjusted"].sum(),
            1,
        ),
        (
            "County-hours with weather",
            integrated_sample["weather_available"].sum(),
            600,
        ),
        (
            "Weather proxy county-hours",
            integrated_sample["weather_proxy_used"].sum(),
            120,
        ),
        (
            "Raw NOAA 999 wind-direction reports",
            integrated_sample["wind_direction_special_count"].sum(),
            199,
        ),
        (
            "Missing aggregated wind-speed hours",
            integrated_sample["wind_speed_m_s"].isna().sum(),
            2,
        ),
        (
            "Missing aggregated wind-direction hours",
            integrated_sample["wind_direction_degrees"].isna().sum(),
            109,
        ),
        (
            "County-hours with all measurements",
            integrated_sample["all_measurements_available"].sum(),
            480,
        ),
    ],
    columns=["check", "observed", "expected"],
)

validation_results["status"] = validation_results.apply(
    lambda row: "PASS" if row["observed"] == row["expected"] else "FAIL",
    axis=1,
)

display(validation_results)

failed_checks = validation_results[
    validation_results["status"] != "PASS"
]

assert failed_checks.empty, (
    "One or more integration checks failed:\n"
    + failed_checks.to_string(index=False)
)

county_grid_validation = (
    integrated_sample.groupby("county", as_index=False)
    .agg(
        county_hour_rows=("event_hour_utc", "size"),
        unique_hours=("event_hour_utc", "nunique"),
        fire_detection_hours=("fire_detection_present", "sum"),
        pm25_available_hours=("pm25_available", "sum"),
        weather_available_hours=("weather_available", "sum"),
        weather_proxy_hours=("weather_proxy_used", "sum"),
    )
)

display(county_grid_validation)

assert (county_grid_validation["county_hour_rows"] == 120).all()
assert (county_grid_validation["unique_hours"] == 120).all()

print("All integration validation checks passed.")

,check,observed,expected,status
0,Integrated county-hour rows,600,600,PASS
1,Duplicate county-hour keys,0,0,PASS
2,NASA structural rows,600,600,PASS
3,EPA structural rows,600,600,PASS
4,NOAA structural rows,600,600,PASS
5,Total fire detections preserved,976,976,PASS
6,County-hours with fire detections,19,19,PASS
7,County-hours with PM2.5,480,480,PASS
8,County-hours without PM2.5,120,120,PASS
9,Negative PM2.5 values adjusted,1,1,PASS


,county,county_hour_rows,unique_hours,fire_detection_hours,pm25_available_hours,weather_available_hours,weather_proxy_hours
0,Butte,120,120,2,120,120,0
1,Lassen,120,120,2,0,120,0
2,Plumas,120,120,15,120,120,120
3,Shasta,120,120,0,120,120,0
4,Tehama,120,120,0,120,120,0


All integration validation checks passed.


In [54]:
from pathlib import Path

integrated_output_directory = (
    fire_input_path.parent.parent / "integrated"
)
integrated_output_directory.mkdir(parents=True, exist_ok=True)

integrated_output_path = (
    integrated_output_directory
    / "dixie_integrated_hourly_county_sample_2021-07-13_17.csv"
)

final_integrated_sample = (
    integrated_sample
    .sort_values(["event_hour_utc", "county"])
    .reset_index(drop=True)
)

final_integrated_sample.to_csv(
    integrated_output_path,
    index=False,
)

print("Integrated dataset saved successfully.")
print("Output path:", integrated_output_path)
print("Saved rows:", len(final_integrated_sample))
print("Saved columns:", len(final_integrated_sample.columns))

Integrated dataset saved successfully.
Output path: /Users/VV/Documents/GitHub/smokewatch-replay/data/interim/integrated/dixie_integrated_hourly_county_sample_2021-07-13_17.csv
Saved rows: 600
Saved columns: 32


In [55]:
integrated_reloaded = pd.read_csv(integrated_output_path)

integrated_reloaded["event_hour_utc"] = pd.to_datetime(
    integrated_reloaded["event_hour_utc"],
    utc=True,
    errors="coerce",
)

boolean_columns = [
    "fire_row_present",
    "pm25_row_present",
    "weather_row_present",
    "fire_detection_present",
    "pm25_available",
    "pm25_negative_adjusted",
    "weather_available",
    "weather_proxy_used",
    "all_measurements_available",
]

def parse_saved_boolean(value):
    if pd.isna(value):
        return False
    return str(value).strip().lower() in {"true", "1", "yes"}

for column in boolean_columns:
    integrated_reloaded[column] = (
        integrated_reloaded[column].map(parse_saved_boolean)
    )

reload_validation = pd.DataFrame(
    {
        "check": [
            "Reloaded rows",
            "Reloaded columns",
            "Invalid timestamps",
            "Duplicate county-hour keys",
            "Counties",
            "Unique hours",
            "Fire detections preserved",
            "PM2.5 available hours",
            "Weather available hours",
        ],
        "observed": [
            len(integrated_reloaded),
            len(integrated_reloaded.columns),
            integrated_reloaded["event_hour_utc"].isna().sum(),
            integrated_reloaded.duplicated(
                subset=["event_hour_utc", "county"]
            ).sum(),
            integrated_reloaded["county"].nunique(),
            integrated_reloaded["event_hour_utc"].nunique(),
            integrated_reloaded["fire_detection_count"].sum(),
            integrated_reloaded["pm25_available"].sum(),
            integrated_reloaded["weather_available"].sum(),
        ],
    }
)

display(reload_validation)
display(integrated_reloaded.head())

assert len(integrated_reloaded) == 600
assert integrated_reloaded["event_hour_utc"].isna().sum() == 0
assert (
    integrated_reloaded.duplicated(
        subset=["event_hour_utc", "county"]
    ).sum()
    == 0
)
assert integrated_reloaded["fire_detection_count"].sum() == 976
assert integrated_reloaded["pm25_available"].sum() == 480
assert integrated_reloaded["weather_available"].sum() == 600

print("Saved integrated dataset reloaded and verified successfully.")

,check,observed
0,Reloaded rows,600
1,Reloaded columns,32
2,Invalid timestamps,0
3,Duplicate county-hour keys,0
4,Counties,5
5,Unique hours,120
6,Fire detections preserved,976
7,PM2.5 available hours,480
8,Weather available hours,600


,event_hour_utc,county,fire_detection_count,total_frp_mw,mean_frp_mw,max_frp_mw,high_confidence_count,night_detection_count,fire_row_present,pm25_ug_m3_raw,...,relative_humidity_percent,wind_speed_m_s,maximum_wind_speed_m_s,wind_direction_degrees,wind_direction_special_count,weather_available,weather_proxy_used,weather_row_present,fire_detection_present,all_measurements_available
0,2021-07-13 00:00:00+00:00,Butte,0,0.0,NaN,NaN,0,0,True,13.000000,...,25.000000,4.10,4.1,170.000000,0,True,False,True,False,True
1,2021-07-13 00:00:00+00:00,Lassen,0,0.0,NaN,NaN,0,0,True,NaN,...,3.333333,7.70,8.2,286.670437,0,True,False,True,False,False
2,2021-07-13 00:00:00+00:00,Plumas,0,0.0,NaN,NaN,0,0,True,0.333333,...,3.333333,7.70,8.2,286.670437,0,True,True,True,False,True
3,2021-07-13 00:00:00+00:00,Shasta,0,0.0,NaN,NaN,0,0,True,7.000000,...,8.000000,6.20,6.2,150.000000,0,True,False,True,False,True
4,2021-07-13 00:00:00+00:00,Tehama,0,0.0,NaN,NaN,0,0,True,7.000000,...,11.500000,5.95,6.2,155.000000,0,True,False,True,False,True


Saved integrated dataset reloaded and verified successfully.


### Sample-Period Feasibility Conclusion

The three-source county-hour integration is feasible for the Dixie Fire pilot.

- NASA FIRMS, EPA AirData, and NOAA GHCNh align on a complete 600-row structural grid covering five counties and 120 UTC hours.
- The join preserves all 976 qualifying fire detections.
- PM2.5 measurements are available for 480 county-hours. The 120 missing hours all belong to Lassen County and remain explicitly missing rather than being converted to zero.
- Weather data are available for all 600 county-hours.
- Plumas County uses 120 hours from the nearby Susanville station as an explicitly labeled weather proxy.
- NOAA wind-direction code `999` is treated as unavailable or variable direction, not as a physical direction.
- Zero fire detections represent no qualifying detection in that county-hour; they do not imply continuous satellite observation.

These results support expanding the pipelines to the full July 13–August 31, 2021 pilot period before implementing the replay and delayed-record experiments.